# 08 — Transit Parameter Estimation

**Objective:** Estimate scientifically meaningful physical parameters for every detected exoplanet candidate using transit modelling, MCMC uncertainty quantification, and publication-quality visualisation.

**Pipeline Stage:** Final scientific notebook — refines candidate parameters from TLS/BLS detection into physical planetary properties.


In [48]:
!pip install -q batman-package emcee corner pandas numpy scipy astropy matplotlib plotly lightkurve joblib tqdm


In [49]:
import gc
import json
import logging
import warnings
import time
import os
from pathlib import Path
from datetime import datetime
from typing import Optional, Tuple, List, Dict, Any

import numpy as np
import pandas as pd
import scipy.optimize as opt
from scipy.stats import chi2
from astropy import units as u
from astropy.timeseries import BoxLeastSquares

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
import matplotlib.patheffects as pe

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import batman
import emcee
import corner

from tqdm.notebook import tqdm
import joblib
from concurrent.futures import ThreadPoolExecutor, as_completed
import multiprocessing

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("Notebook08")


In [50]:
GPU_AVAILABLE = False
try:
    import cupy as cp
    GPU_AVAILABLE = cp.cuda.is_available()
    if GPU_AVAILABLE:
        mem = cp.cuda.runtime.memGetInfo()
        logger.info(f"GPU: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")
        logger.info(f"GPU memory: {mem[1] // 1024**2} MB total, {mem[0] // 1024**2} MB free")
        cp.get_default_memory_pool()
except Exception:
    pass
logger.info(f"CPU cores: {multiprocessing.cpu_count()}")
logger.info(f"GPU: {'ENABLED' if GPU_AVAILABLE else 'NOT AVAILABLE'}")


2026-07-28 00:39:46,479 | INFO | GPU: NVIDIA GeForce RTX 3050 Laptop GPU
2026-07-28 00:39:46,479 | INFO | GPU memory: 4095 MB total, 3305 MB free
2026-07-28 00:39:46,479 | INFO | CPU cores: 16
2026-07-28 00:39:46,479 | INFO | GPU: ENABLED


In [51]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
LIGHTCURVE_DIR = DATA_DIR / "lightcurves"
CANDIDATE_DIR = DATA_DIR / "candidates"
FEATURE_DIR = DATA_DIR / "explainability"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = PROJECT_ROOT / "figures"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
O_CANDIDATE = OUTPUTS_DIR / "candidate_parameters"
O_MODELS = OUTPUTS_DIR / "transit_models"
O_FIGURES = OUTPUTS_DIR / "figures"
O_REPORTS = OUTPUTS_DIR / "reports"
O_MCMC = OUTPUTS_DIR / "mcmc"
O_LOGS = OUTPUTS_DIR / "logs"

for d in [OUTPUTS_DIR, O_CANDIDATE, O_MODELS, O_FIGURES, O_REPORTS, O_MCMC, O_LOGS]:
    d.mkdir(parents=True, exist_ok=True)

logger.info("Directories created under outputs/")


2026-07-28 00:39:46,518 | INFO | Directories created under outputs/


In [52]:
def load_inputs():
    inputs = {}
    paths = {
        "candidate_catalog": CANDIDATE_DIR / "candidate_catalog.parquet",
        "feature_table": FEATURE_DIR / "feature_table.parquet",
        "master_catalog": DATA_DIR / "master_catalog.parquet",
        "feature_columns": MODELS_DIR / "feature_columns.json",
    }
    for name, path in paths.items():
        if path.exists():
            if path.suffix == ".parquet":
                inputs[name] = pd.read_parquet(path)
            elif path.suffix == ".json":
                with open(path) as f:
                    inputs[name] = json.load(f)
            logger.info(f"Loaded {name}: {path}")
        else:
            logger.warning(f"Missing {name}: {path}")
            inputs[name] = None

    # Try loading AI predictions and probabilities
    for fname in ["prediction_probabilities.csv", "classification_report.txt"]:
        p = REPORTS_DIR / "explainability" / fname
        if p.exists():
            if fname.endswith(".csv"):
                inputs[fname.replace(".csv", "")] = pd.read_csv(p)
            logger.info(f"Loaded {fname}")

    lc_files = sorted(LIGHTCURVE_DIR.glob("TIC_*.parquet"))
    inputs["lightcurve_files"] = lc_files
    logger.info(f"Found {len(lc_files)} lightcurve files")

    return inputs

data = load_inputs()


2026-07-28 00:39:46,672 | INFO | Loaded candidate_catalog: d:\MachineLearning\Exoplanet-detection\data\processed\candidates\candidate_catalog.parquet
2026-07-28 00:39:46,678 | INFO | Loaded feature_table: d:\MachineLearning\Exoplanet-detection\data\processed\explainability\feature_table.parquet
2026-07-28 00:39:46,696 | INFO | Loaded master_catalog: d:\MachineLearning\Exoplanet-detection\data\processed\master_catalog.parquet
2026-07-28 00:39:46,696 | INFO | Loaded feature_columns: d:\MachineLearning\Exoplanet-detection\models\feature_columns.json
2026-07-28 00:39:46,696 | INFO | Loaded prediction_probabilities.csv
2026-07-28 00:39:46,706 | INFO | Found 89 lightcurve files


In [53]:
cc = data.get("candidate_catalog")
ft = data.get("feature_table")
mc = data.get("master_catalog")
lc_files = data.get("lightcurve_files", [])
feature_cols = data.get("feature_columns")

if cc is None:
    raise FileNotFoundError("candidate_catalog required")

logger.info(f"Candidates: {len(cc)}")
logger.info(f"PASS: {(cc.quality_flag=='PASS').sum()}, REVIEW: {(cc.quality_flag=='REVIEW').sum()}, REJECT: {(cc.quality_flag=='REJECT').sum()}")
if ft is not None:
    logger.info(f"Feature table: {len(ft)} rows, {ft.columns.tolist()[-3:]}")
if mc is not None:
    logger.info(f"Master catalog: {len(mc)} stars, cols={list(mc.columns)}")


2026-07-28 00:39:46,746 | INFO | Candidates: 89
2026-07-28 00:39:46,750 | INFO | PASS: 7, REVIEW: 10, REJECT: 72
2026-07-28 00:39:46,751 | INFO | Feature table: 89 rows, ['tic_id', 'label', 'class_name']
2026-07-28 00:39:46,752 | INFO | Master catalog: 6028 stars, cols=['tic_id', 'planet_name', 'host_star', 'ra', 'dec', 'orbital_period_days', 'planet_radius_earth', 'planet_mass_earth', 'stellar_teff', 'stellar_radius', 'stellar_mass', 'distance_pc', 'discovery_year', 'class_label', 'class_id', 'source_catalog']


In [54]:
def get_lightcurve(tic_id, sector=None):
    if sector is not None:
        candidates = [f for f in lc_files if f"TIC_{tic_id}_Sector_{sector}" in f.name]
    else:
        candidates = [f for f in lc_files if f"TIC_{tic_id}" in f.name]
    if not candidates:
        logger.warning(f"No lightcurve for TIC {tic_id}")
        return None
    df = pd.read_parquet(candidates[0])
    if "trend" in df.columns:
        df["flux_detrended"] = df["flux"] / df["trend"]
    else:
        df["flux_detrended"] = df["flux"]
    df["flux_norm"] = df["flux_detrended"] / np.nanmedian(df["flux_detrended"])
    return df

def validate_lightcurve(df, min_points=100):
    if df is None:
        return False
    if len(df) < min_points:
        return False
    if np.isnan(df["time"]).all() or np.isnan(df["flux_norm"]).all():
        return False
    return True

# Quick test
test_lc = get_lightcurve(cc.iloc[0]["tic_id"], cc.iloc[0]["sector"])
if test_lc is not None:
    logger.info(f"Test LC: {len(test_lc)} pts, time=[{test_lc.time.min():.1f}, {test_lc.time.max():.1f}]")


2026-07-28 00:39:46,807 | INFO | Test LC: 13532 pts, time=[1386.0, 1406.2]


# Part 2 — Candidate Selection

Select high-quality candidates (PASS + REVIEW) for transit parameter estimation.


In [55]:
def select_candidates(candidate_df, feature_df=None, min_sde=5.0):
    selected = candidate_df[candidate_df["quality_flag"].isin(["PASS", "REVIEW"])].copy()
    selected = selected[selected["sde"] >= min_sde].copy()
    selected = selected.sort_values("sde", ascending=False).reset_index(drop=True)
    if feature_df is not None and "tic_id" in feature_df.columns:
        selected = selected.merge(
            feature_df[["tic_id", "label", "class_name"]], on="tic_id", how="left"
        )
    if mc is not None:
        selected = selected.merge(
            mc[["tic_id", "stellar_teff", "stellar_radius", "stellar_mass", "planet_radius_earth", "orbital_period_days"]],
            on="tic_id", how="left"
        )
    logger.info(f"Selected {len(selected)} candidates for transit modelling")
    return selected

candidates = select_candidates(cc, ft)
candidates[["tic_id", "period", "duration", "depth", "sde", "quality_flag", "class_name", "distinct_transit_count"]]


2026-07-28 00:39:46,832 | INFO | Selected 23 candidates for transit modelling


,tic_id,period,duration,depth,sde,quality_flag,class_name,distinct_transit_count
0,460396820,2.339798,0.038277,0.989643,16.932605,PASS,Planet,4
1,158002130,9.697182,0.113098,0.998686,16.578593,PASS,Planet,2
2,152476657,3.611938,0.112233,0.994994,16.561641,PASS,Planet,6
3,288246496,4.302186,0.138474,0.993620,16.245796,PASS,Planet,7
4,27318774,3.243644,0.107481,0.986707,14.959567,PASS,Planet,8
5,262662119,4.532162,0.082501,0.989921,14.811831,PASS,Planet,3
6,375654303,1.801054,0.115733,0.993536,14.674431,PASS,Planet,13
7,273373582,0.607383,0.061321,0.992550,11.085018,REVIEW,Planet,43
8,272080244,13.282089,0.087017,0.966113,9.535652,REVIEW,Planet,2
9,100990000,9.581046,0.088920,0.999720,9.485989,REVIEW,Planet,2


In [56]:
logger.info(f"Candidate distribution by class:")
class_counts = candidates.groupby("class_name").size()
for cls, cnt in class_counts.items():
    logger.info(f"  {cls}: {cnt}")
logger.info(f"Quality flags: PASS={(candidates.quality_flag=='PASS').sum()}, REVIEW={(candidates.quality_flag=='REVIEW').sum()}")


2026-07-28 00:39:46,868 | INFO | Candidate distribution by class:
2026-07-28 00:39:46,871 | INFO |   Planet: 23
2026-07-28 00:39:46,872 | INFO | Quality flags: PASS=7, REVIEW=16


In [57]:
def crossmatch_with_master_catalog(candidates_df, master_df):
    if master_df is None:
        return candidates_df, []
    matched = candidates_df.merge(
        master_df[["tic_id", "planet_name", "orbital_period_days", "planet_radius_earth",
                   "stellar_teff", "stellar_radius", "stellar_mass", "discovery_year"]],
        on="tic_id", how="left"
    )
    known = matched[matched["planet_name"].notna()]
    unknown = matched[matched["planet_name"].isna()]
    if len(known) > 0:
        logger.info(f"Cross-matched {len(known)} candidates with known planets")
    return matched, known[["tic_id", "planet_name"]].values.tolist() if len(known) > 0 else []

candidates_matched, known_planets = crossmatch_with_master_catalog(candidates, mc)
if len(known_planets) > 0:
    logger.info("Known planets in candidate set:")
    for tid, name in known_planets:
        logger.info(f"  TIC {tid}: {name}")


2026-07-28 00:39:46,911 | INFO | Cross-matched 39 candidates with known planets
2026-07-28 00:39:46,911 | INFO | Known planets in candidate set:
2026-07-28 00:39:46,911 | INFO |   TIC 460396820: WASP-24 b
2026-07-28 00:39:46,911 | INFO |   TIC 158002130: TOI-1180 b
2026-07-28 00:39:46,911 | INFO |   TIC 152476657: WASP-120 b
2026-07-28 00:39:46,911 | INFO |   TIC 288246496: WASP-60 b
2026-07-28 00:39:46,911 | INFO |   TIC 27318774: Kepler-485 b
2026-07-28 00:39:46,911 | INFO |   TIC 262662119: WASP-151 b
2026-07-28 00:39:46,921 | INFO |   TIC 375654303: TOI-4034 b
2026-07-28 00:39:46,922 | INFO |   TIC 273373582: Kepler-1229 b
2026-07-28 00:39:46,923 | INFO |   TIC 272080244: Kepler-1738 b
2026-07-28 00:39:46,923 | INFO |   TIC 100990000: TOI-411 b
2026-07-28 00:39:46,923 | INFO |   TIC 100990000: HD 22946 d
2026-07-28 00:39:46,923 | INFO |   TIC 100990000: TOI-411 c
2026-07-28 00:39:46,923 | INFO |   TIC 100990000: TOI-411 b
2026-07-28 00:39:46,923 | INFO |   TIC 100990000: HD 22946 d

In [58]:
def compute_statistics_per_candidate(row):
    tic_id = int(row["tic_id"])
    lc = get_lightcurve(tic_id, int(row.get("sector", 0)))
    if lc is None: return {}
    stats = {
        "tic_id": tic_id,
        "n_points": len(lc),
        "time_span": lc["time"].max() - lc["time"].min(),
        "median_flux": float(np.nanmedian(lc["flux_norm"])),
        "rms_flux": float(np.nanstd(lc["flux_norm"])),
        "cadence_min": float(np.nanmedian(np.diff(lc["time"])) * 24 * 60),
    }
    n_transits = int(np.floor(stats["time_span"] / row["period"]))
    stats["expected_transits"] = n_transits
    stats["observed_ratio"] = min(1.0, row.get("distinct_transit_count", 0) / max(n_transits, 1))
    return stats

candidate_stats = []
for _, row in candidates.iterrows():
    s = compute_statistics_per_candidate(row)
    if s: candidate_stats.append(s)
stats_df = pd.DataFrame(candidate_stats) if candidate_stats else pd.DataFrame()
if len(stats_df) > 0:
    stats_df[["tic_id", "n_points", "expected_transits", "observed_ratio", "cadence_min"]]


In [59]:
if len(stats_df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist(stats_df["n_points"]/1000, bins=12, color="steelblue", edgecolor="k")
    axes[0].set_xlabel("Points (K)"); axes[0].set_ylabel("Count"); axes[0].set_title("Light Curve Length")
    axes[1].hist(stats_df["observed_ratio"], bins=12, color="coral", edgecolor="k")
    axes[1].set_xlabel("Observed/Expected Transits"); axes[1].set_ylabel("Count")
    axes[1].set_title("Transit Coverage Ratio")
    axes[2].scatter(stats_df["time_span"], stats_df["expected_transits"], c=stats_df["observed_ratio"],
                    s=40, cmap="viridis", edgecolors="k", alpha=0.7)
    axes[2].set_xlabel("Time Span (days)"); axes[2].set_ylabel("Expected Transits")
    axes[2].set_title("Transit Coverage")
    cb = plt.colorbar(axes[2].collections[0], ax=axes[2]); cb.set_label("Obs/Exp Ratio")
    plt.tight_layout()
    plt.savefig(O_FIGURES / "candidate_statistics.png", dpi=150, bbox_inches="tight")
    plt.show()


In [60]:
logger.info("\n" + "="*70)
logger.info(f"{'TIC_ID':<12} {'Period':>8} {'SDE':>7} {'Flag':<8} {'Class':<20} {'Depth':>8} {'Dur':>6}")
logger.info("="*70)
for _, row in candidates.iterrows():
    logger.info(f"{int(row['tic_id']):<12} {row['period']:>8.3f} {row['sde']:>7.2f} "
                f"{str(row['quality_flag']):<8} {str(row.get('class_name','?')):<20} "
                f"{row['depth']:>8.4f} {row['duration']:>6.4f}")
logger.info("="*70)


2026-07-28 00:39:48,065 | INFO | 
2026-07-28 00:39:48,073 | INFO | TIC_ID         Period     SDE Flag     Class                   Depth    Dur
2026-07-28 00:39:48,073 | INFO | ======================================================================
2026-07-28 00:39:48,075 | INFO | 460396820       2.340   16.93 PASS     Planet                 0.9896 0.0383
2026-07-28 00:39:48,075 | INFO | 158002130       9.697   16.58 PASS     Planet                 0.9987 0.1131
2026-07-28 00:39:48,078 | INFO | 152476657       3.612   16.56 PASS     Planet                 0.9950 0.1122
2026-07-28 00:39:48,078 | INFO | 288246496       4.302   16.25 PASS     Planet                 0.9936 0.1385
2026-07-28 00:39:48,080 | INFO | 27318774        3.244   14.96 PASS     Planet                 0.9867 0.1075
2026-07-28 00:39:48,080 | INFO | 262662119       4.532   14.81 PASS     Planet                 0.9899 0.0825
2026-07-28 00:39:48,082 | INFO | 375654303       1.801   14.67 PASS     Planet                 0.99

In [61]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

axes[0].bar(range(len(candidates)), candidates["sde"], color="steelblue", edgecolor="navy")
axes[0].axhline(7, color="red", ls="--", alpha=0.5, label="SDE=7 threshold")
axes[0].set_xlabel("Candidate Index"); axes[0].set_ylabel("SDE"); axes[0].set_title("Signal Detection Efficiency"); axes[0].legend()

axes[1].scatter(candidates["period"], candidates["sde"], c=candidates["quality_flag"].map({"PASS":"green","REVIEW":"orange"}),
                s=80, edgecolors="k", alpha=0.8)
axes[1].set_xlabel("Period (days)"); axes[1].set_ylabel("SDE"); axes[1].set_title("Period vs SDE")
axes[1].set_xscale("log")

axes[2].hist(candidates["period"], bins=15, color="steelblue", edgecolor="k", alpha=0.7)
axes[2].set_xlabel("Period (days)"); axes[2].set_ylabel("Count"); axes[2].set_title("Period Distribution")

depths = 1 - candidates["depth"]
axes[3].scatter(candidates["period"], depths * 1e6, c=candidates["sde"], s=60, cmap="viridis", edgecolors="k", alpha=0.8)
axes[3].set_xlabel("Period (days)"); axes[3].set_ylabel("Depth (ppm)"); axes[3].set_title("Depth vs Period")
axes[3].set_xscale("log"); axes[3].set_yscale("log")
cb = plt.colorbar(axes[3].collections[0], ax=axes[3]); cb.set_label("SDE")

plt.tight_layout()
plt.savefig(O_FIGURES / "candidate_selection.png", dpi=150, bbox_inches="tight")
plt.show()


In [62]:
def plot_candidate_lightcurve(tic_id, sector, period, epoch, duration):
    lc = get_lightcurve(tic_id, sector)
    if lc is None: return
    phase = ((lc["time"] - epoch + 0.5 * period) % period / period) - 0.5
    fold_mask = np.abs(phase) < 2 * duration / period * 3

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), gridspec_kw={"height_ratios":[1,1.2]})

    ax1.plot(lc["time"], lc["flux_norm"], "k.", ms=1, alpha=0.4)
    ax1.axvline(epoch, color="r", ls="--", alpha=0.6, label=f"T0={epoch:.2f}")
    for n in range(-2, 3):
        ax1.axvline(epoch + n * period, color="orange", ls=":", alpha=0.4)
    ax1.set_xlabel("Time (BTJD)"); ax1.set_ylabel("Normalized Flux"); ax1.set_title(f"TIC {tic_id} — Sector {sector}")
    ax1.legend(fontsize=8)

    ax2.plot(phase[fold_mask], lc["flux_norm"][fold_mask], "k.", ms=1.5, alpha=0.3)
    bin_edges = np.linspace(-0.5, 0.5, 101)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    binned_flux = np.array([np.nanmedian(lc["flux_norm"][fold_mask][np.digitize(phase[fold_mask], bin_edges[:-1]) == i])
                           for i in range(1, len(bin_edges))])
    ax2.plot(bin_centers, binned_flux, "r-", lw=2)
    ax2.axvline(0, color="gray", ls="--", alpha=0.3)
    ax2.set_xlabel("Phase"); ax2.set_ylabel("Normalized Flux"); ax2.set_title("Phase-folded light curve")
    ax2.set_xlim(-0.5, 0.5)

    plt.tight_layout()
    plt.savefig(O_FIGURES / f"lc_tic{tic_id}.png", dpi=150, bbox_inches="tight")
    plt.show()

# Show first few candidates
for _, row in candidates.head(4).iterrows():
    plot_candidate_lightcurve(int(row["tic_id"]), int(row["sector"]), row["period"], row["epoch"], row["duration"])


# Part 3 — Initial Transit Parameter Estimation

Estimate physical transit parameters from TLS/BLS detection results.


In [63]:
def estimate_initial_parameters(row):
    params = {}
    params["tic_id"] = int(row["tic_id"])
    params["sector"] = int(row["sector"])

    period = float(row["period"])
    epoch = float(row["epoch"])
    duration = float(row["duration"])
    depth = float(row["depth"])
    depth_ppm = (1 - depth) * 1e6

    params["period"] = period
    params["epoch"] = epoch
    params["duration"] = duration
    params["duration_hours"] = duration * 24
    params["depth"] = depth
    params["depth_ppm"] = depth_ppm
    params["sde"] = float(row["sde"])
    params["transit_snr"] = float(row.get("transit_snr", 0))

    Rp_Rs = np.sqrt(1 - depth) if depth < 1 else 0.1
    params["rp_rs"] = Rp_Rs

    if duration > 0 and period > 0:
        a_Rs = np.sqrt((1 + Rp_Rs)**2 - (0.0)**2) / (np.pi * duration / period)
        if a_Rs < 1: a_Rs = 5.0
        params["a_rs"] = float(a_Rs)
    else:
        params["a_rs"] = 5.0

    params["inclination"] = 90.0
    params["impact_parameter"] = 0.0

    params["stellar_teff"] = float(row.get("stellar_teff", 5778) or 5778)
    params["stellar_radius"] = float(row.get("stellar_radius", 1.0) or 1.0)
    params["stellar_mass"] = float(row.get("stellar_mass", 1.0) or 1.0)

    if params["stellar_teff"] > 0:
        params["equilibrium_temp"] = float(params["stellar_teff"] * np.sqrt(1.0 / (2 * params["a_rs"])))
    else:
        params["equilibrium_temp"] = np.nan

    params["transit_probability"] = float(params["stellar_radius"] / (params["a_rs"] * 0.004649)) if params["a_rs"] > 0 else np.nan

    planet_radius = Rp_Rs * params["stellar_radius"]
    params["planet_radius_earth"] = float(planet_radius * 9.731) if planet_radius > 0 else np.nan
    params["planet_radius_jupiter"] = float(planet_radius * 0.102763) if planet_radius > 0 else np.nan

    params["quality_flag"] = str(row["quality_flag"])
    params["detector"] = str(row.get("detector", "TLS"))
    params["class_name"] = str(row.get("class_name", "Unknown"))
    return params

initial_params = []
for _, row in candidates.iterrows():
    initial_params.append(estimate_initial_parameters(row))

params_df = pd.DataFrame(initial_params)
params_df[["tic_id", "period", "rp_rs", "a_rs", "inclination", "planet_radius_earth", "equilibrium_temp", "transit_probability", "depth_ppm", "class_name"]]


,tic_id,period,rp_rs,a_rs,inclination,planet_radius_earth,equilibrium_temp,transit_probability,depth_ppm,class_name
0,460396820,2.339798,0.101769,21.438041,90.0,1.406250,927.766613,14.247665,10356.982364,Planet
1,158002130,9.697182,0.036247,28.281590,90.0,0.257487,629.982012,5.552128,1313.865515,Planet
2,152476657,3.611938,0.070755,10.968841,90.0,1.159755,1377.096404,33.031833,5006.241713,Planet
3,288246496,4.302186,0.079875,10.679316,90.0,0.909397,1276.630734,23.565838,6380.000574,Planet
4,27318774,3.243644,0.115297,10.713763,90.0,1.222928,1287.106524,21.883911,13293.337165,Planet
5,262662119,4.532162,0.100393,19.241633,90.0,1.153744,946.403023,13.202264,10078.696184,Planet
6,375654303,1.801054,0.080401,5.351877,90.0,1.431753,1800.615340,73.550459,6464.265925,Planet
7,273373582,0.607383,0.086315,3.424984,90.0,0.428366,1445.795543,32.029646,7450.332627,Planet
8,272080244,13.282089,0.184085,57.530290,90.0,1.793127,565.228806,3.742639,33887.441164,Planet
9,100990000,9.581046,0.016734,34.871731,90.0,0.181895,738.691363,6.890014,280.040825,Planet


In [64]:
logger.info("Initial parameter estimation complete")
logger.info(f"Period range: [{params_df['period'].min():.3f}, {params_df['period'].max():.3f}] days")

def classify_planet_size(r_earth):
    if r_earth < 1.0: return "Sub-Earth"
    elif r_earth < 1.5: return "Earth-size"
    elif r_earth < 3.0: return "Super-Earth"
    elif r_earth < 6.0: return "Sub-Neptune"
    elif r_earth < 12.0: return "Neptune-size"
    else: return "Jupiter-size"

params_df["size_class"] = params_df["planet_radius_earth"].apply(
    lambda x: classify_planet_size(x) if not np.isnan(x) and x > 0 else "Unknown"
)
size_dist = params_df["size_class"].value_counts()
logger.info("Planet size distribution:")
for sz, cnt in size_dist.items():
    logger.info(f"  {sz}: {cnt}")

def classify_habitability(teq):
    if np.isnan(teq): return "Unknown"
    if 250 < teq < 350: return "Optimistic Habitable"
    elif 180 < teq < 400: return "Conservative Habitable"
    else: return "Non-Habitable"

params_df["habitability"] = params_df["equilibrium_temp"].apply(classify_habitability)
hab_dist = params_df["habitability"].value_counts()
logger.info("Habitability classification:")
for hb, cnt in hab_dist.items():
    logger.info(f"  {hb}: {cnt}")


2026-07-28 00:39:52,703 | INFO | Initial parameter estimation complete
2026-07-28 00:39:52,703 | INFO | Period range: [0.607, 13.282] days
2026-07-28 00:39:52,708 | INFO | Planet size distribution:
2026-07-28 00:39:52,710 | INFO |   Sub-Earth: 15
2026-07-28 00:39:52,710 | INFO |   Earth-size: 7
2026-07-28 00:39:52,710 | INFO |   Super-Earth: 1
2026-07-28 00:39:52,714 | INFO | Habitability classification:
2026-07-28 00:39:52,716 | INFO |   Non-Habitable: 22
2026-07-28 00:39:52,716 | INFO |   Conservative Habitable: 1


In [65]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
size_cls = params_df["size_class"].value_counts()
colors_sz = ["gold", "lightgreen", "steelblue", "coral", "purple", "gray"]
axes[0].bar(size_cls.index, size_cls.values, color=colors_sz[:len(size_cls)], edgecolor="k")
axes[0].set_xlabel("Size Class"); axes[0].set_ylabel("Count")
axes[0].set_title("Planet Size Distribution"); axes[0].tick_params(axis="x", rotation=45)

hab_cls = params_df["habitability"].value_counts()
axes[1].bar(hab_cls.index, hab_cls.values, color=["green", "orange", "red", "gray"], edgecolor="k")
axes[1].set_xlabel("Habitability"); axes[1].set_ylabel("Count")
axes[1].set_title("Habitability Classification"); axes[1].tick_params(axis="x", rotation=45)

axes[2].scatter(params_df["period"], params_df["equilibrium_temp"], c=params_df["planet_radius_earth"],
                s=60, cmap="viridis", edgecolors="k", alpha=0.8)
axes[2].axhline(350, color="orange", ls="--", alpha=0.5, label="350 K")
axes[2].axhline(250, color="green", ls="--", alpha=0.5, label="250 K")
axes[2].set_xlabel("Period (days)"); axes[2].set_ylabel("T_eq (K)")
axes[2].set_title("Habitability Diagram"); axes[2].set_xscale("log"); axes[2].legend()
cb = plt.colorbar(axes[2].collections[0], ax=axes[2]); cb.set_label("R_p (R_Earth)")

plt.tight_layout()
plt.savefig(O_FIGURES / "habitability_diagram.png", dpi=150, bbox_inches="tight")
plt.show()


In [66]:
logger.info(f"Period range: [{params_df['period'].min():.3f}, {params_df['period'].max():.3f}] days")
logger.info(f"Rp/Rs range: [{params_df['rp_rs'].min():.4f}, {params_df['rp_rs'].max():.4f}]")
logger.info(f"a/Rs range: [{params_df['a_rs'].min():.2f}, {params_df['a_rs'].max():.2f}]")
logger.info(f"Planet radius range: [{params_df['planet_radius_earth'].min():.2f}, {params_df['planet_radius_earth'].max():.2f}] R_Earth")


2026-07-28 00:39:53,698 | INFO | Period range: [0.607, 13.282] days
2026-07-28 00:39:53,703 | INFO | Rp/Rs range: [0.0093, 0.1841]
2026-07-28 00:39:53,704 | INFO | a/Rs range: [3.42, 86.03]
2026-07-28 00:39:53,706 | INFO | Planet radius range: [0.09, 1.79] R_Earth


In [67]:
def create_transit_window(t_obs, f_obs, period, epoch, duration, window_mult=3):
    phase = ((t_obs - epoch + 0.5 * period) % period / period) - 0.5
    half_window = window_mult * max(duration, 0.02) / period
    mask = np.abs(phase) < half_window
    return t_obs[mask], f_obs[mask], mask, phase

def compute_phased_lightcurve(t_obs, f_obs, period, epoch, n_bins=200):
    phase = ((t_obs - epoch + 0.5 * period) % period / period) - 0.5
    bins = np.linspace(-0.5, 0.5, n_bins + 1)
    bin_c = 0.5 * (bins[:-1] + bins[1:])
    bin_idx = np.digitize(phase, bins[:-1])
    binned_f = np.array([np.nanmedian(f_obs[bin_idx == i]) for i in range(1, n_bins + 1)])
    binned_e = np.array([np.nanstd(f_obs[bin_idx == i]) / np.sqrt(max((bin_idx == i).sum(), 1))
                         for i in range(1, n_bins + 1)])
    return bin_c, binned_f, binned_e, phase

def estimate_transit_snr(t_obs, f_obs, period, epoch, duration):
    _, zoom_f, _, _ = create_transit_window(t_obs, f_obs, period, epoch, duration, window_mult=1)
    _, out_f, _, _ = create_transit_window(t_obs, f_obs, period, epoch, duration, window_mult=4)
    if len(zoom_f) < 10 or len(out_f) < 10: return 0
    transit_depth = 1 - np.nanmedian(zoom_f)
    out_of_transit_rms = np.nanstd(out_f)
    return transit_depth / out_of_transit_rms if out_of_transit_rms > 0 else 0

# Compute SNR for each candidate
for i, row in candidates.iterrows():
    lc = get_lightcurve(int(row["tic_id"]), int(row["sector"]))
    if lc is None: continue
    snr = estimate_transit_snr(lc["time"].values, lc["flux_norm"].values, row["period"], row["epoch"], row["duration"])
    params_df.at[i, "estimated_snr"] = snr
logger.info("Transit SNR computed for all candidates")


2026-07-28 00:39:54,006 | INFO | Transit SNR computed for all candidates


In [68]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

rp = params_df["planet_radius_earth"].replace([np.inf, -np.inf], np.nan)
axes[0].hist(rp.dropna(), bins=12, color="coral", edgecolor="k", alpha=0.7)
axes[0].axvline(rp.median(), color="r", ls="--", label=f"Median: {rp.median():.1f} R_Earth")
axes[0].set_xlabel("Planet Radius (R_Earth)"); axes[0].set_ylabel("Count"); axes[0].set_title("Initial Planet Radius"); axes[0].legend()

axes[1].hist(params_df["a_rs"].replace([np.inf, -np.inf], np.nan).dropna(), bins=12, color="lightgreen", edgecolor="k", alpha=0.7)
axes[1].set_xlabel("a/R_s"); axes[1].set_ylabel("Count"); axes[1].set_title("Semi-major Axis Ratio")

axes[2].hist(params_df["equilibrium_temp"].replace([np.inf, -np.inf], np.nan).dropna(), bins=12, color="gold", edgecolor="k", alpha=0.7)
axes[2].set_xlabel("T_eq (K)"); axes[2].set_ylabel("Count"); axes[2].set_title("Equilibrium Temperature")

d = params_df["depth_ppm"]
axes[3].hist(d.replace([np.inf, -np.inf], np.nan).dropna(), bins=15, color="steelblue", edgecolor="k", alpha=0.7)
axes[3].set_xlabel("Depth (ppm)"); axes[3].set_ylabel("Count"); axes[3].set_title("Transit Depth")

axes[4].scatter(params_df["period"], params_df["planet_radius_earth"], c=params_df["sde"],
                s=60, cmap="plasma", edgecolors="k", alpha=0.8)
axes[4].set_xlabel("Period (days)"); axes[4].set_ylabel("Planet Radius (R_Earth)")
axes[4].set_title("Radius vs Period"); axes[4].set_xscale("log")
cb = plt.colorbar(axes[4].collections[0], ax=axes[4]); cb.set_label("SDE")

axes[5].scatter(params_df["a_rs"], params_df["equilibrium_temp"], c=params_df["transit_probability"],
                s=60, cmap="viridis", edgecolors="k", alpha=0.8)
axes[5].set_xlabel("a/R_s"); axes[5].set_ylabel("T_eq (K)"); axes[5].set_title("T_eq vs Semi-major Axis")
cb2 = plt.colorbar(axes[5].collections[0], ax=axes[5]); cb2.set_label("Transit Probability")

plt.tight_layout()
plt.savefig(O_FIGURES / "initial_parameters.png", dpi=150, bbox_inches="tight")
plt.show()


# Part 4 — BATMAN Transit Modelling

Generate synthetic transit light curves using BATMAN and fit them to observed data.


In [69]:
def create_batman_model(params, t):
    ma = batman.TransitParams()
    ma.t0 = float(params["epoch"])
    ma.per = float(params["period"])
    ma.rp = float(params["rp_rs"])
    ma.a = float(params["a_rs"])
    ma.inc = float(params["inclination"])
    ma.ecc = 0.0
    ma.w = 90.0
    ma.u = [0.4, 0.2]
    ma.limb_dark = "quadratic"
    ma.fp = 0.0
    ma.t_secondary = 0.0
    return ma

def generate_transit_model(ma, t, supersample_factor=3):
    if supersample_factor > 1:
        m = batman.TransitModel(ma, t, supersample_factor=supersample_factor, exp_time=0.5/24/60)
    else:
        m = batman.TransitModel(ma, t)
    flux_model = m.light_curve(ma)
    return flux_model, m

def compute_chi2(f_obs, f_model, ferr=None):
    residual = f_obs - f_model
    if ferr is not None:
        return np.sum((residual / ferr) ** 2)
    return np.sum(residual ** 2)

def compute_bic(chi2, n_params, n_data): return chi2 + n_params * np.log(n_data)
def compute_aic(chi2, n_params, n_data): return chi2 + 2 * n_params
def compute_aicc(chi2, n_params, n_data):
    return compute_aic(chi2, n_params, n_data) + 2 * n_params * (n_params + 1) / (n_data - n_params - 1)

def fit_transit_lightcurve(tic_id, sector, initial_params, t_obs, f_obs, ferr_obs=None):
    period = initial_params["period"]
    epoch = initial_params["epoch"]
    depth = initial_params["depth"]
    duration = initial_params["duration"]
    rp_rs = initial_params["rp_rs"]
    a_rs = initial_params["a_rs"]

    def model_flux(params):
        p, t0, rp, inc, a = params
        ma = batman.TransitParams()
        ma.t0 = t0; ma.per = p; ma.rp = rp; ma.a = a; ma.inc = inc
        ma.ecc = 0.0; ma.w = 90.0; ma.u = [0.4, 0.2]; ma.limb_dark = "quadratic"
        ma.fp = 0.0; ma.t_secondary = 0.0
        m = batman.TransitModel(ma, t_obs)
        return m.light_curve(ma)

    def residuals(params):
        return f_obs - model_flux(params)

    def chisq(params):
        r = residuals(params)
        if ferr_obs is not None:
            return np.sum((r / ferr_obs) ** 2)
        return np.sum(r ** 2)

    bounds = [
        (period * 0.98, period * 1.02),
        (epoch - 0.5, epoch + 0.5),
        (max(rp_rs * 0.5, 0.001), min(rp_rs * 1.5, 0.5)),
        (75.0, 90.0),
        (max(a_rs * 0.8, 1.5), a_rs * 1.2),
    ]
    x0 = [period, epoch, rp_rs, initial_params["inclination"], a_rs]

    result = None
    best_method = "none"
    history = {}
    for method in ["L-BFGS-B", "Powell", "Nelder-Mead"]:
        try:
            res = opt.minimize(chisq, x0, method=method, bounds=bounds,
                               options={"maxiter": 5000, "ftol": 1e-10})
            history[method] = {"success": res.success, "fun": res.fun, "nit": res.nit,
                               "x": res.x.tolist(), "message": res.message}
            if result is None or res.fun < result.fun:
                result = res
                best_method = method
        except Exception as e:
            history[method] = {"success": False, "error": str(e)}

    best = result.x if result is not None else x0
    best_model = model_flux(best)
    chi2_val = chisq(best)
    dof = len(f_obs) - len(best)
    red_chi2 = chi2_val / dof if dof > 0 else np.inf
    rmse = np.sqrt(np.mean((f_obs - best_model) ** 2))
    mae = np.mean(np.abs(f_obs - best_model))
    bic = chi2_val + len(best) * np.log(len(f_obs))
    aic = chi2_val + 2 * len(best) + 2 * len(best) * (len(best) + 1) / (len(f_obs) - len(best) - 1)

    return {
        "tic_id": tic_id,
        "period": best[0], "epoch": best[1], "rp_rs": best[2],
        "inclination": best[3], "a_rs": best[4],
        "model_flux": best_model,
        "chi2": chi2_val, "red_chi2": red_chi2, "rmse": rmse, "mae": mae,
        "bic": bic, "aic": aic,
        "dof": dof, "n_obs": len(f_obs),
        "optimization_history": history,
        "success": result is not None and result.success,
        "best_method": best_method,
    }


In [70]:
def batman_model_quick(params, t):
    ma = batman.TransitParams()
    ma.t0 = params[1]; ma.per = params[0]; ma.rp = params[2]
    ma.inc = params[3]; ma.a = params[4]
    ma.ecc = 0.0; ma.w = 90.0; ma.u = [0.4, 0.2]
    ma.limb_dark = "quadratic"; ma.fp = 0.0; ma.t_secondary = 0.0
    m = batman.TransitModel(ma, t)
    return m.light_curve(ma)

def objective_function(params, t, f):
    model = batman_model_quick(params, t)
    return np.sum((f - model) ** 2)

def chi2_function(params, t, f, ferr):
    model = batman_model_quick(params, t)
    return np.sum(((f - model) / ferr) ** 2)


In [71]:
def process_candidate(row):
    tic_id = int(row["tic_id"])
    sector = int(row["sector"])
    lc = get_lightcurve(tic_id, sector)
    if not validate_lightcurve(lc):
        return None

    t_obs = lc["time"].values.astype(np.float64)
    f_obs = lc["flux_norm"].values.astype(np.float64)
    ferr = lc.get("flux_err", pd.Series(np.std(f_obs) * np.ones_like(f_obs))).values.astype(np.float64)

    init = estimate_initial_parameters(row)

    # Fold and zoom around transit
    period = init["period"]
    epoch = init["epoch"]
    duration = init["duration"]
    phase = ((t_obs - epoch + 0.5 * period) % period / period) - 0.5
    transit_window = 2.0 * max(duration, 0.02)
    zoom_mask = np.abs(phase) < transit_window

    if zoom_mask.sum() < 50:
        zoom_mask = np.ones_like(t_obs, dtype=bool)

    t_zoom = t_obs[zoom_mask]
    f_zoom = f_obs[zoom_mask]
    ferr_zoom = ferr[zoom_mask] if ferr is not None else None

    result = fit_transit_lightcurve(tic_id, sector, init, t_zoom, f_zoom, ferr_zoom)
    result["initial_params"] = init
    result["t_zoom"] = t_zoom
    result["f_zoom"] = f_zoom
    result["t_full"] = t_obs
    result["f_full"] = f_obs
    result["phase_full"] = phase

    period_best = result["period"]
    epoch_best = result["epoch"]
    folded_best = ((t_obs - epoch_best + 0.5 * period_best) % period_best / period_best) - 0.5
    result["phase_best"] = folded_best

    # Compute physical parameters
    rp_rs = result["rp_rs"]
    stellar_r = init.get("stellar_radius", 1.0)
    result["planet_radius_rjup"] = rp_rs * stellar_r * 0.102763
    result["planet_radius_rearth"] = rp_rs * stellar_r * 9.731
    result["transit_depth_pct"] = (1 - (1 - rp_rs ** 2)) * 100
    result["impact_parameter"] = result["a_rs"] * np.cos(np.radians(result["inclination"]))
    inc_rad = np.radians(result["inclination"])
    result["transit_duration_hours"] = (result["period"] / np.pi) * np.arcsin(
        np.sqrt((1 + rp_rs) ** 2 - result["impact_parameter"] ** 2) /
        (result["a_rs"] * np.sin(inc_rad))
    ) * 24 if result["a_rs"] * np.sin(inc_rad) > 0 else duration * 24

    teff = init.get("stellar_teff", 5778)
    result["equilibrium_temp"] = teff * np.sqrt(1.0 / (2 * result["a_rs"])) if result["a_rs"] > 0 else np.nan
    result["transit_probability_pct"] = (init["stellar_radius"] / (result["a_rs"] * 0.004649)) * 100

    return result

# Process first candidate as test
test_result = process_candidate(candidates.iloc[0])
if test_result is not None:
    logger.info(f"Test fit: TIC {test_result['tic_id']}, chi2={test_result['red_chi2']:.3f}, "
                f"RMSE={test_result['rmse']:.6f}, method={test_result['best_method']}")


2026-07-28 00:39:56,522 | INFO | Test fit: TIC 460396820, chi2=0.000, RMSE=0.004027, method=Nelder-Mead


In [72]:
def plot_transit_fit(result):
    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    fig.suptitle(f"TIC {result['tic_id']} — Transit Model Fit", fontsize=14, fontweight="bold")

    # Full light curve
    axes[0, 0].plot(result["t_full"], result["f_full"], "k.", ms=0.8, alpha=0.3)
    zoom_t = result["t_zoom"]
    phase = result["phase_full"]
    zoom_idx = np.abs(phase) < 2 * result.get("duration", 0.1) / result["period"] * 3
    axes[0, 0].plot(result["t_full"][zoom_idx], result["f_full"][zoom_idx], "r.", ms=1, alpha=0.5)
    axes[0, 0].set_xlabel("Time (BTJD)"); axes[0, 0].set_ylabel("Normalized Flux")
    axes[0, 0].set_title("Full Light Curve (zoom region in red)")

    # Zoomed transit
    axes[0, 1].plot(zoom_t, result["f_zoom"], "k.", ms=2, alpha=0.5)
    axes[0, 1].plot(zoom_t, result["model_flux"], "r-", lw=2, label="BATMAN fit")
    axes[0, 1].set_xlabel("Time (BTJD)"); axes[0, 1].set_ylabel("Flux")
    axes[0, 1].set_title("Zoomed Transit + Model"); axes[0, 1].legend()

    # Phase folded
    axes[1, 0].plot(result["phase_full"], result["f_full"], "k.", ms=0.8, alpha=0.2)
    bin_e = np.linspace(-0.5, 0.5, 151)
    bc = 0.5 * (bin_e[:-1] + bin_e[1:])
    bidx = np.digitize(result["phase_full"], bin_e[:-1])
    bin_f = np.array([np.nanmedian(result["f_full"][bidx == i]) for i in range(1, len(bin_e))])
    axes[1, 0].plot(bc, bin_f, "b-", lw=2, alpha=0.8)
    axes[1, 0].set_xlabel("Phase"); axes[1, 0].set_ylabel("Flux")
    axes[1, 0].set_title("Phase-folded Light Curve"); axes[1, 0].set_xlim(-0.5, 0.5)

    # Residuals
    residual = result["f_zoom"] - result["model_flux"]
    axes[1, 1].plot(zoom_t, residual, "k.", ms=2, alpha=0.5)
    axes[1, 1].axhline(0, color="r", ls="--", lw=1)
    axes[1, 1].set_xlabel("Time (BTJD)"); axes[1, 1].set_ylabel("Residual")
    axes[1, 1].set_title(f"Residuals (RMSE={result['rmse']:.6f})")
    ylim = np.nanpercentile(np.abs(residual), 99) * 1.2
    axes[1, 1].set_ylim(-ylim, ylim)

    # Residual histogram
    axes[2, 0].hist(residual, bins=30, color="steelblue", edgecolor="k", alpha=0.7)
    axes[2, 0].axvline(0, color="r", ls="--")
    axes[2, 0].set_xlabel("Residual"); axes[2, 0].set_ylabel("Count")
    axes[2, 0].set_title(f"Residual Distribution (σ={np.std(residual):.6f})")

    # Parameter table
    axes[2, 1].axis("off")
    text_data = [
        ["Parameter", "Initial", "Fitted"],
        ["Period (d)", f"{result['initial_params']['period']:.4f}", f"{result['period']:.4f}"],
        ["Epoch", f"{result['initial_params']['epoch']:.2f}", f"{result['epoch']:.2f}"],
        ["Rp/Rs", f"{result['initial_params']['rp_rs']:.4f}", f"{result['rp_rs']:.4f}"],
        ["Inclination", f"{result['initial_params']['inclination']:.1f}", f"{result['inclination']:.1f}"],
        ["a/Rs", f"{result['initial_params']['a_rs']:.2f}", f"{result['a_rs']:.2f}"],
        ["R_p (R_Earth)", f"{result['initial_params'].get('planet_radius_earth', np.nan):.2f}",
         f"{result['planet_radius_rearth']:.2f}"],
        ["T_eq (K)", f"{result['initial_params'].get('equilibrium_temp', np.nan):.0f}",
         f"{result['equilibrium_temp']:.0f}"],
        ["χ²_red", "", f"{result['red_chi2']:.3f}"],
    ]
    table = axes[2, 1].table(cellText=text_data, loc="center", cellLoc="center")
    table.auto_set_font_size(False); table.set_fontsize(9)
    table.scale(1, 1.5)
    for j in range(3):
        table[0, j].set_facecolor("navy")
        table[0, j].set_text_props(color="white", fontweight="bold")

    plt.tight_layout()
    plt.savefig(O_FIGURES / f"transit_fit_{result['tic_id']}.png", dpi=150, bbox_inches="tight")
    plt.show()

# Display transit fit for first candidate
if test_result is not None:
    plot_transit_fit(test_result)


In [73]:
def compare_initial_vs_fitted(initial_df, result_row):
    comparison = pd.DataFrame({
        "Parameter": ["Period", "Epoch", "Rp/Rs", "Inclination", "a/Rs"],
        "Initial": [initial_df["period"], initial_df["epoch"], initial_df["rp_rs"],
                    initial_df["inclination"], initial_df["a_rs"]],
        "Fitted": [result_row["period"], result_row["epoch"], result_row["rp_rs"],
                   result_row["inclination"], result_row["a_rs"]],
        "Change %": [
            (result_row["period"] - initial_df["period"]) / initial_df["period"] * 100,
            (result_row["epoch"] - initial_df["epoch"]) / initial_df["epoch"] * 100,
            (result_row["rp_rs"] - initial_df["rp_rs"]) / initial_df["rp_rs"] * 100,
            (result_row["inclination"] - initial_df["inclination"]) / initial_df["inclination"] * 100,
            (result_row["a_rs"] - initial_df["a_rs"]) / initial_df["a_rs"] * 100,
        ]
    })
    return comparison

if test_result is not None:
    comp = compare_initial_vs_fitted(test_result["initial_params"], test_result)
    comp


In [74]:
def fit_all_candidates(candidate_rows, max_workers=4):
    results = []
    failures = []
    def process(row):
        try:
            return process_candidate(row)
        except Exception as e:
            return {"tic_id": int(row["tic_id"]), "error": str(e), "success": False}

    if len(candidate_rows) == 0:
        return results, failures

    with ThreadPoolExecutor(max_workers=min(max_workers, len(candidate_rows))) as ex:
        fut_map = {ex.submit(process, row): i for i, row in candidate_rows.iterrows()}
        for f in tqdm(as_completed(fut_map), total=len(fut_map), desc="Fitting transits"):
            res = f.result()
            if res is None:
                failures.append({"tic_id": candidate_rows.iloc[fut_map[f]]["tic_id"], "error": "No LC"})
            elif "error" in res:
                failures.append(res)
            else:
                results.append(res)

    df = pd.DataFrame(results) if results else pd.DataFrame()
    fail_df = pd.DataFrame(failures) if failures else pd.DataFrame()
    return df, fail_df

transit_results, transit_failures = fit_all_candidates(candidates, max_workers=4)
logger.info(f"Fit {len(transit_results)} candidates, {len(transit_failures)} failures")
if len(transit_results) > 0:
    transit_results[["tic_id", "period", "rp_rs", "inclination", "a_rs", "red_chi2", "rmse"]]


Fitting transits:   0%|          | 0/23 [00:00<?, ?it/s]

2026-07-28 00:40:21,544 | INFO | Fit 23 candidates, 0 failures


In [75]:
def analyze_fit_residuals(transit_df):
    residuals_list = []
    for _, row in transit_df.iterrows():
        if "f_zoom" not in row or "model_flux" not in row: continue
        res = row["f_zoom"] - row["model_flux"]
        residuals_list.append({
            "tic_id": int(row["tic_id"]),
            "residual_mean": float(np.mean(res)),
            "residual_std": float(np.std(res)),
            "residual_skew": float(pd.Series(res).skew()),
            "residual_kurtosis": float(pd.Series(res).kurtosis()),
            "residual_rms": float(np.sqrt(np.mean(res**2))),
            "max_residual": float(np.max(np.abs(res))),
            "n_outliers_3sigma": int(np.sum(np.abs(res) > 3*np.std(res))),
        })
    return pd.DataFrame(residuals_list)

residual_analysis = analyze_fit_residuals(transit_results) if len(transit_results) > 0 else pd.DataFrame()
if len(residual_analysis) > 0:
    residual_analysis[["tic_id", "residual_mean", "residual_std", "residual_rms", "n_outliers_3sigma"]]


In [76]:
if len(residual_analysis) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(residual_analysis["residual_std"], bins=12, color="steelblue", edgecolor="k")
    axes[0].set_xlabel("Residual Std Dev"); axes[0].set_ylabel("Count"); axes[0].set_title("Residual Scatter")
    n_out = residual_analysis["n_outliers_3sigma"]
    axes[1].scatter(residual_analysis["residual_rms"], n_out, c=n_out, s=40, cmap="hot", edgecolors="k", alpha=0.7)
    axes[1].set_xlabel("RMS Residual"); axes[1].set_ylabel("3σ Outliers"); axes[1].set_title("Outlier Analysis")
    plt.tight_layout()
    plt.savefig(O_FIGURES / "residual_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()


In [77]:
logger.info(f"Batch transit fitting complete:")
logger.info(f"  Success: {len(transit_results)}")
logger.info(f"  Failures: {len(transit_failures)}")
if len(transit_failures) > 0:
    logger.warning(f"  Failed TICs: {[int(f.get('tic_id',0)) for f in transit_failures]}")


2026-07-28 00:40:22,139 | INFO | Batch transit fitting complete:
2026-07-28 00:40:22,142 | INFO |   Success: 23
2026-07-28 00:40:22,143 | INFO |   Failures: 0


In [78]:
if len(transit_results) > 0:
    transit_results.to_parquet(O_MODELS / "transit_fits.parquet", index=False)
    transit_results.to_csv(O_MODELS / "transit_fits.csv", index=False)
    logger.info(f"Saved {len(transit_results)} transit fits")


2026-07-28 00:40:22,531 | INFO | Saved 23 transit fits


In [79]:
def analyze_per_candidate_fit_quality(transit_df):
    results = []
    for _, row in transit_df.iterrows():
        tic_id = int(row["tic_id"])
        chi2 = row.get("red_chi2", np.nan)
        r2 = row.get("r2", np.nan)
        rmse = row.get("rmse", np.nan)
        n_obs = row.get("n_obs", 0)
        if not np.isnan(chi2):
            if chi2 < 0.5: fit_quality = "Overfit"
            elif chi2 < 1.5: fit_quality = "Good"
            elif chi2 < 3: fit_quality = "Acceptable"
            else: fit_quality = "Poor"
        else: fit_quality = "Unknown"
        results.append({"tic_id": tic_id, "fit_quality": fit_quality, "red_chi2": chi2,
                        "r2": r2, "rmse": rmse, "n_obs": n_obs})
    return pd.DataFrame(results)

fit_quality_df = analyze_per_candidate_fit_quality(transit_results)
if len(fit_quality_df) > 0:
    qual_counts = fit_quality_df["fit_quality"].value_counts()
    logger.info("Per-candidate fit quality:")
    for q, c in qual_counts.items(): logger.info(f"  {q}: {c}")
    fit_quality_df


2026-07-28 00:40:22,594 | INFO | Per-candidate fit quality:
2026-07-28 00:40:22,595 | INFO |   Overfit: 23


# Part 5 — Parameter Optimisation

Compare optimisation methods and compute goodness-of-fit statistics.


In [80]:
def compare_optimization_methods(t_obs, f_obs, initial_params):
    results = []
    for method in ["L-BFGS-B", "Powell", "Nelder-Mead", "TNC", "SLSQP"]:
        try:
            def objective(x):
                p, t0, rp, inc, a = x
                ma = batman.TransitParams()
                ma.t0 = t0; ma.per = p; ma.rp = rp; ma.a = a; ma.inc = inc
                ma.ecc = 0.0; ma.w = 90.0; ma.u = [0.4, 0.2]; ma.limb_dark = "quadratic"
                ma.fp = 0.0; ma.t_secondary = 0.0
                m = batman.TransitModel(ma, t_obs)
                model = m.light_curve(ma)
                return np.sum((f_obs - model) ** 2)

            bounds = [
                (initial_params["period"] * 0.95, initial_params["period"] * 1.05),
                (initial_params["epoch"] - 0.5, initial_params["epoch"] + 0.5),
                (0.001, 0.5),
                (75.0, 90.0),
                (1.5, 30.0),
            ]
            x0 = [initial_params["period"], initial_params["epoch"],
                  initial_params["rp_rs"], initial_params["inclination"], initial_params["a_rs"]]

            t0 = time.time()
            res = opt.minimize(objective, x0, method=method, bounds=bounds,
                               options={"maxiter": 5000, "ftol": 1e-12})
            elapsed = time.time() - t0

            results.append({
                "method": method,
                "success": res.success,
                "fun": res.fun,
                "nfev": res.nfev,
                "nit": res.nit,
                "time_s": elapsed,
                "x": res.x.tolist(),
                "message": res.message,
            })
        except Exception as e:
            results.append({"method": method, "success": False, "error": str(e)})

    return pd.DataFrame(results)

# Compare on first candidate
if test_result is not None:
    t_zoom = test_result["t_zoom"]
    f_zoom = test_result["f_zoom"]
    init = test_result["initial_params"]
    comp_df = compare_optimization_methods(t_zoom, f_zoom, init)
    comp_df


In [81]:
def plot_optimization_convergence(history_dict, tic_id):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    methods = list(history_dict.keys())
    fun_vals = [history_dict[m].get("fun", np.nan) for m in methods if history_dict[m].get("success")]
    nfev_vals = [history_dict[m].get("nfev", 0) for m in methods if history_dict[m].get("success")]
    labels = [m for m in methods if history_dict.get(m, {}).get("success")]

    if len(fun_vals) > 0:
        axes[0].bar(labels, fun_vals, color="steelblue", edgecolor="k")
        axes[0].set_xlabel("Method"); axes[0].set_ylabel("Final χ²"); axes[0].set_title("Chi-Square by Method")
        axes[0].tick_params(axis="x", rotation=45)

        axes[1].bar(labels, nfev_vals, color="coral", edgecolor="k")
        axes[1].set_xlabel("Method"); axes[1].set_ylabel("Function Evaluations")
        axes[1].set_title("Computational Cost by Method"); axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.savefig(O_FIGURES / f"optimization_convergence_{tic_id}.png", dpi=150, bbox_inches="tight")
    plt.show()

if test_result is not None and "optimization_history" in test_result:
    plot_optimization_convergence(test_result["optimization_history"], test_result["tic_id"])


In [82]:
def compute_goodness_of_fit(f_obs, f_model, n_params):
    residual = f_obs - f_model
    chi2 = np.sum(residual ** 2 / (np.std(f_obs - f_model) ** 2 + 1e-30))
    dof = len(f_obs) - n_params
    red_chi2 = chi2 / dof if dof > 0 else np.inf
    rmse = np.sqrt(np.mean(residual ** 2))
    mae = np.mean(np.abs(residual))
    bic = chi2 + n_params * np.log(len(f_obs))
    aic = chi2 + 2 * n_params
    aicc = aic + 2 * n_params * (n_params + 1) / (len(f_obs) - n_params - 1)
    r2 = 1 - np.sum(residual ** 2) / np.sum((f_obs - np.mean(f_obs)) ** 2)
    return {
        "chi2": chi2, "dof": dof, "red_chi2": red_chi2,
        "rmse": rmse, "mae": mae, "bic": bic, "aic": aic,
        "aicc": aicc, "r2": r2, "residual_std": np.std(residual),
    }

if len(transit_results) > 0:
    for i, row in transit_results.iterrows():
        if "model_flux" in row and "f_zoom" in row:
            gof = compute_goodness_of_fit(row["f_zoom"], row["model_flux"], 5)
            transit_results.at[i, "red_chi2"] = gof["red_chi2"]
            transit_results.at[i, "rmse"] = gof["rmse"]
            transit_results.at[i, "bic"] = gof["bic"]
            transit_results.at[i, "aic"] = gof["aic"]
            transit_results.at[i, "r2"] = gof["r2"]
            transit_results.at[i, "mae"] = gof["mae"]

    summary_cols = ["tic_id", "period", "rp_rs", "red_chi2", "rmse", "bic", "aic", "r2"]
    transit_results[[c for c in summary_cols if c in transit_results.columns]]


In [83]:
def compute_parameter_uncertainty_from_fit(transit_df):
    uncert = []
    for _, row in transit_df.iterrows():
        if "optimization_history" not in row or not row.get("success", False):
            continue
        try:
            hist = row["optimization_history"]
            best_method = row.get("best_method", "L-BFGS-B")
            if best_method in hist and hist[best_method].get("success"):
                import scipy.optimize as opt
                t_zoom = row.get("t_zoom")
                f_zoom = row.get("f_zoom")
                if t_zoom is None or f_zoom is None: continue
                def obj_fn(x):
                    p, t0, rp, inc, a = x
                    ma = batman.TransitParams()
                    ma.t0 = t0; ma.per = p; ma.rp = rp; ma.a = a; ma.inc = inc
                    ma.ecc = 0; ma.w = 90; ma.u = [0.4, 0.2]; ma.limb_dark = "quadratic"
                    ma.fp = 0; ma.t_secondary = 0
                    m = batman.TransitModel(ma, t_zoom)
                    return np.sum((f_zoom - m.light_curve(ma))**2)
                x0 = [row["period"], row["epoch"], row["rp_rs"], row["inclination"], row["a_rs"]]
                res_hess = opt.minimize(obj_fn, x0, method="L-BFGS-B")
                if hasattr(res_hess, "hess_inv"):
                    try:
                        cov = res_hess.hess_inv.todense()
                        errors = np.sqrt(np.diag(cov))
                        uncert.append({"tic_id": int(row["tic_id"]),
                            "period_err": errors[0], "epoch_err": errors[1],
                            "rp_rs_err": errors[2], "inc_err": errors[3], "a_rs_err": errors[4]})
                    except: pass
        except: pass
    return pd.DataFrame(uncert)

param_uncertainty = compute_parameter_uncertainty_from_fit(transit_results)
if len(param_uncertainty) > 0:
    logger.info("Parameter uncertainties from Hessian approximation:")
    param_uncertainty


2026-07-28 00:40:26,037 | INFO | Parameter uncertainties from Hessian approximation:


In [84]:
# Optimisation summary plot
if len(transit_results) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    axes[0].bar(range(len(transit_results)), transit_results["red_chi2"].fillna(1), color="steelblue", edgecolor="k")
    axes[0].axhline(1, color="r", ls="--", alpha=0.7, label="χ²_red = 1")
    axes[0].set_xlabel("Candidate Index"); axes[0].set_ylabel("Reduced χ²")
    axes[0].set_title("Goodness of Fit"); axes[0].legend()

    axes[1].scatter(transit_results["rmse"], transit_results["bic"], c=transit_results["r2"],
                    s=60, cmap="viridis", edgecolors="k", alpha=0.8)
    axes[1].set_xlabel("RMSE"); axes[1].set_ylabel("BIC")
    axes[1].set_title("RMSE vs BIC")
    cb = plt.colorbar(axes[1].collections[0], ax=axes[1]); cb.set_label("R²")

    axes[2].hist(transit_results["r2"].fillna(0), bins=12, color="coral", edgecolor="k", alpha=0.7)
    axes[2].set_xlabel("R²"); axes[2].set_ylabel("Count"); axes[2].set_title("R² Distribution")

    axes[3].axis("off")
    table_data = [["Metric", "Mean", "Median", "Std"],
                  ["χ²_red", f"{transit_results['red_chi2'].mean():.3f}",
                   f"{transit_results['red_chi2'].median():.3f}",
                   f"{transit_results['red_chi2'].std():.3f}"],
                  ["RMSE", f"{transit_results['rmse'].mean():.6f}",
                   f"{transit_results['rmse'].median():.6f}",
                   f"{transit_results['rmse'].std():.6f}"],
                  ["R²", f"{transit_results['r2'].mean():.4f}",
                   f"{transit_results['r2'].median():.4f}",
                   f"{transit_results['r2'].std():.4f}"],
                  ["BIC", f"{transit_results['bic'].mean():.0f}",
                   f"{transit_results['bic'].median():.0f}",
                   f"{transit_results['bic'].std():.0f}"]]
    t = axes[3].table(cellText=table_data, loc="center", cellLoc="center")
    t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1, 1.5)
    for j in range(4): t[0, j].set_facecolor("navy"); t[0, j].set_text_props(color="white", fontweight="bold")

    plt.tight_layout()
    plt.savefig(O_FIGURES / "optimization_summary.png", dpi=150, bbox_inches="tight")
    plt.show()


# Part 6 — Uncertainty Estimation with MCMC

Use emcee to sample posterior distributions and estimate parameter uncertainties.


In [85]:
def run_mcmc(t_obs, f_obs, ferr_obs, initial_params, n_walkers=32, n_steps=2000, n_burn=500):
    n_params = 5
    param_names = ["period", "epoch", "rp_rs", "inclination", "a_rs"]
    ndim = n_params

    def log_prior(params):
        p, t0, rp, inc, a = params
        if not (0.1 < p < 50): return -np.inf
        if not (t0 - 10 < t0 < t0 + 10): return -np.inf
        inc_rad = np.radians(inc)
        if not (0.001 < rp < 0.5): return -np.inf
        if not (70 < inc < 90): return -np.inf
        if not (1.5 < a < 30): return -np.inf
        return 0.0

    def log_likelihood(params):
        p, t0, rp, inc, a = params
        try:
            ma = batman.TransitParams()
            ma.t0 = t0; ma.per = p; ma.rp = rp; ma.a = a; ma.inc = inc
            ma.ecc = 0.0; ma.w = 90.0; ma.u = [0.4, 0.2]; ma.limb_dark = "quadratic"
            ma.fp = 0.0; ma.t_secondary = 0.0
            m = batman.TransitModel(ma, t_obs)
            model = m.light_curve(ma)
            sigma2 = ferr_obs ** 2 + model ** 2 * np.exp(2 * 0.01)
            return -0.5 * np.sum((f_obs - model) ** 2 / sigma2 + np.log(2 * np.pi * sigma2))
        except Exception:
            return -np.inf

    def log_probability(params):
        lp = log_prior(params)
        if not np.isfinite(lp):
            return -np.inf
        return lp + log_likelihood(params)

    p_init = [initial_params["period"], initial_params["epoch"],
              initial_params["rp_rs"], initial_params["inclination"], initial_params["a_rs"]]
    scales = [p_init[0] * 0.01, 0.05, p_init[2] * 0.1, 2.0, p_init[4] * 0.1]
    scales = [max(s, 1e-4) for s in scales]
    pos = [p_init + scales * np.random.randn(ndim) for _ in range(n_walkers)]

    sampler = emcee.EnsembleSampler(n_walkers, ndim, log_probability)
    sampler.run_mcmc(pos, n_steps, progress=True)

    samples = sampler.get_chain(discard=n_burn, flat=True)
    log_prob_samples = sampler.get_log_prob(discard=n_burn, flat=True)

    mcmc_result = {
        "samples": samples,
        "log_prob": log_prob_samples,
        "sampler": sampler,
        "param_names": param_names,
        "n_walkers": n_walkers,
        "n_steps": n_steps,
        "n_burn": n_burn,
    }

    # Compute medians and percentiles
    for i, name in enumerate(param_names):
        mcmc_result[f"{name}_median"] = np.median(samples[:, i])
        mcmc_result[f"{name}_p16"] = np.percentile(samples[:, i], 16)
        mcmc_result[f"{name}_p84"] = np.percentile(samples[:, i], 84)

    return mcmc_result

def plot_mcmc_diagnostics(mcmc_result, tic_id):
    sampler = mcmc_result["sampler"]
    fig, axes = plt.subplots(5, 2, figsize=(14, 12))
    fig.suptitle(f"TIC {tic_id} — MCMC Diagnostics", fontsize=14, fontweight="bold")

    n_steps = mcmc_result["n_steps"]
    n_burn = mcmc_result["n_burn"]

    for i in range(5):
        ax = axes[i, 0]
        for j in range(min(mcmc_result["n_walkers"], 16)):
            ax.plot(sampler.chain[j, :, i], "k", alpha=0.1, lw=0.5)
        ax.axvline(n_burn, color="r", ls="--", alpha=0.7)
        ax.set_ylabel(mcmc_result["param_names"][i])
        if i == 4: ax.set_xlabel("Step")

        ax2 = axes[i, 1]
        samples = mcmc_result["samples"][:, i]
        ax2.hist(samples, bins=50, density=True, color="steelblue", edgecolor="k", alpha=0.7)
        ax2.axvline(mcmc_result[f"{mcmc_result['param_names'][i]}_median"], color="r", ls="--", lw=2, label="Median")
        ax2.axvline(mcmc_result[f"{mcmc_result['param_names'][i]}_p16"], color="orange", ls=":", lw=1.5, label="16th")
        ax2.axvline(mcmc_result[f"{mcmc_result['param_names'][i]}_p84"], color="orange", ls=":", lw=1.5, label="84th")
        ax2.set_ylabel("Density")
        if i == 0: ax2.legend(fontsize=7)
        if i == 4: ax2.set_xlabel("Value")

    plt.tight_layout()
    plt.savefig(O_FIGURES / f"mcmc_diagnostics_{tic_id}.png", dpi=150, bbox_inches="tight")
    plt.show()

def plot_corner(mcmc_result, tic_id):
    samples = mcmc_result["samples"]
    labels = [r"$P$ (days)", r"$T_0$ (BTJD)", r"$R_p/R_s$", r"$i$ (deg)", r"$a/R_s$"]
    truths = [mcmc_result[f"{n}_median"] for n in mcmc_result["param_names"]]

    fig = corner.corner(samples, labels=labels, truths=truths, quantiles=[0.16, 0.5, 0.84],
                        show_titles=True, title_kwargs={"fontsize": 10},
                        plot_datapoints=False, fill_contours=True,
                        levels=(1 - np.exp(-0.5), 1 - np.exp(-2), 1 - np.exp(-9/2)),
                        color="steelblue", bins=30, smooth=1.0,
                        title_fmt=".4f", max_n_ticks=4)
    fig.suptitle(f"TIC {tic_id} — Posterior Distribution", fontsize=14, fontweight="bold")
    plt.savefig(O_FIGURES / f"corner_{tic_id}.png", dpi=300, bbox_inches="tight")
    plt.show()


In [86]:
def prepare_mcmc_data(tic_id, period, epoch, rp_rs, inclination, a_rs):
    lc = get_lightcurve(tic_id)
    if lc is None: return None, None, None, None
    t_obs = lc["time"].values.astype(np.float64)
    f_obs = lc["flux_norm"].values.astype(np.float64)
    ferr = lc.get("flux_err", pd.Series(np.std(f_obs) * np.ones_like(f_obs))).values.astype(np.float64)
    phase = ((t_obs - epoch + 0.5 * period) % period / period) - 0.5
    zoom = np.abs(phase) < 0.1
    if zoom.sum() < 50: zoom = np.ones_like(t_obs, dtype=bool)
    return t_obs[zoom], f_obs[zoom], ferr[zoom], {
        "period": period, "epoch": epoch, "rp_rs": rp_rs,
        "inclination": inclination, "a_rs": a_rs
    }

def compute_autocorrelation_time(sampler, discard=100):
    try:
        tau = emcee.autocorr.integrated_time(sampler.get_chain(discard=discard), tol=0)
        return tau
    except Exception:
        return np.array([np.nan] * sampler.ndim)

def compute_gelman_rubin(sampler, n_chains=None, discard=100):
    chains = sampler.get_chain(discard=discard)
    n, m, d = chains.shape
    if n_chains is not None:
        chains = chains[:, :n_chains, :]; m = n_chains
    chain_mean = np.mean(chains, axis=0)
    overall_mean = np.mean(chain_mean, axis=0)
    B = m * np.var(chain_mean, axis=0, ddof=1)
    W = np.mean(np.var(chains, axis=0, ddof=1), axis=0)
    var_est = (n - 1) / n * W + B / n
    R_hat = np.sqrt(var_est / W)
    return R_hat

# Run MCMC on best candidate
if len(transit_results) > 0:
    best_idx = transit_results["red_chi2"].fillna(999).idxmin()
    best_row = transit_results.iloc[best_idx]
    tic_id = int(best_row["tic_id"])

    lc = get_lightcurve(tic_id)
    if lc is not None:
        t_obs = lc["time"].values.astype(np.float64)
        f_obs = (lc["flux_norm"]).values.astype(np.float64)
        ferr = lc.get("flux_err", pd.Series(np.std(f_obs) * np.ones_like(f_obs))).values.astype(np.float64)

        init = {
            "period": float(best_row["period"]),
            "epoch": float(best_row["epoch"]),
            "rp_rs": float(best_row["rp_rs"]),
            "inclination": float(best_row["inclination"]),
            "a_rs": float(best_row["a_rs"]),
        }

        # Fold and zoom
        period = init["period"]
        epoch = init["epoch"]
        phase = ((t_obs - epoch + 0.5 * period) % period / period) - 0.5
        zoom = np.abs(phase) < 0.1
        if zoom.sum() < 50: zoom = np.ones_like(t_obs, dtype=bool)

        logger.info(f"Running MCMC for TIC {tic_id} with {zoom.sum()} points")

        mcmc_result = run_mcmc(t_obs[zoom], f_obs[zoom], ferr[zoom], init,
                               n_walkers=32, n_steps=2000, n_burn=500)
        logger.info(f"MCMC done: acceptance={mcmc_result['sampler'].acceptance_fraction.mean():.3f}")

        def analyze_mcmc_convergence(mcmc_result):
            sampler = mcmc_result["sampler"]
            chains = sampler.get_chain()
            n_steps, n_walkers, n_dim = chains.shape
            # Gelman-Rubin statistic
            chain_means = np.mean(chains[int(n_steps*0.5):], axis=0)
            overall_mean = np.mean(chain_means, axis=0)
            B = n_steps * 0.5 * np.var(chain_means, axis=0)
            W = np.mean(np.var(chains[int(n_steps*0.5):], axis=0, ddof=1), axis=0)
            R_hat = np.sqrt((1 - 1/n_steps) + B/(W * n_steps * n_walkers))
            for i, name in enumerate(mcmc_result["param_names"]):
                logger.info(f"  R_hat({name}) = {R_hat[i]:.4f}")
            return R_hat

        r_hat = analyze_mcmc_convergence(mcmc_result)
        if np.all(r_hat < 1.1):
            logger.info("MCMC convergence: GOOD (all R_hat < 1.1)")
        else:
            logger.warning("MCMC convergence: POOR (some R_hat >= 1.1)")

        plot_mcmc_diagnostics(mcmc_result, tic_id)
        plot_corner(mcmc_result, tic_id)

        # Posterior statistics
        posterior_summary = []
        for i, name in enumerate(mcmc_result["param_names"]):
            posterior_summary.append({
                "parameter": name,
                "median": mcmc_result[f"{name}_median"],
                "p16": mcmc_result[f"{name}_p16"],
                "p84": mcmc_result[f"{name}_p84"],
                "std": np.std(mcmc_result["samples"][:, i]),
            })
        post_df = pd.DataFrame(posterior_summary)
        logger.info("Posterior statistics:")
        logger.info(post_df.to_string(index=False))

        # Parameter correlations from MCMC
        samples_arr = mcmc_result["samples"]
        corr_matrix = np.corrcoef(samples_arr.T)
        corr_df = pd.DataFrame(corr_matrix, index=mcmc_result["param_names"], columns=mcmc_result["param_names"])
        logger.info(f"Parameter correlations:\n{corr_df.to_string()}")
        corr_df.to_csv(O_MCMC / f"parameter_correlations_{tic_id}.csv")

        # Save MCMC samples
        samples_df = pd.DataFrame(mcmc_result["samples"], columns=mcmc_result["param_names"])
        samples_df["log_prob"] = mcmc_result["log_prob"]
        samples_df.to_parquet(O_MCMC / f"mcmc_samples_{tic_id}.parquet", index=False)
        logger.info(f"MCMC samples saved: {len(samples_df)}")
        
        # Compute convergence diagnostics
        try:
            autocorr = compute_autocorrelation_time(mcmc_result["sampler"])
            logger.info(f"Autocorrelation times: {autocorr}")
            r_hat = compute_gelman_rubin(mcmc_result["sampler"])
            logger.info(f"Gelman-Rubin R_hat: {r_hat}")
        except Exception as e:
            logger.warning(f"Convergence diagnostics failed: {e}")


2026-07-28 00:40:27,246 | INFO | Running MCMC for TIC 138219758 with 3777 points
100%|██████████| 2000/2000 [00:24<00:00, 82.65it/s]
2026-07-28 00:40:51,466 | INFO | MCMC done: acceptance=0.259
2026-07-28 00:40:51,469 | INFO |   R_hat(period) = 1.0023
2026-07-28 00:40:51,469 | INFO |   R_hat(epoch) = 1.0018
2026-07-28 00:40:51,469 | INFO |   R_hat(rp_rs) = 1.0021
2026-07-28 00:40:51,469 | INFO |   R_hat(inclination) = 1.0024
2026-07-28 00:40:51,469 | INFO |   R_hat(a_rs) = 1.0025
2026-07-28 00:40:51,469 | INFO | MCMC convergence: GOOD (all R_hat < 1.1)
2026-07-28 00:40:56,253 | INFO | Posterior statistics:
2026-07-28 00:40:56,268 | INFO |   parameter        median           p16          p84          std
     period     29.430640  8.599927e+00 4.540643e+01 1.538901e+01
      epoch -12942.127338 -4.339712e+14 2.732455e+15 3.809154e+16
      rp_rs      0.313670  9.962972e-02 4.557241e-01 1.507289e-01
inclination     80.429403  7.288375e+01 8.753007e+01 6.074488e+00
       a_rs     11.3585

In [87]:
def run_mcmc_batch(transit_results_df, max_candidates=5, n_walkers=24, n_steps=1500, n_burn=400):
    mcmc_all = []
    for _, row in transit_results_df.head(max_candidates).iterrows():
        tic_id = int(row["tic_id"])
        logger.info(f"--- MCMC for TIC {tic_id} ---")
        lc = get_lightcurve(tic_id)
        if lc is None:
            mcmc_all.append({"tic_id": tic_id, "error": "No lightcurve"})
            continue
        try:
            t_obs = lc["time"].values.astype(np.float64)
            f_obs = lc["flux_norm"].values.astype(np.float64)
            ferr = lc.get("flux_err", pd.Series(np.std(f_obs) * np.ones_like(f_obs))).values.astype(np.float64)
            init = {"period": float(row["period"]), "epoch": float(row["epoch"]),
                    "rp_rs": float(row["rp_rs"]), "inclination": float(row["inclination"]),
                    "a_rs": float(row["a_rs"])}
            period = init["period"]; epoch = init["epoch"]
            phase = ((t_obs - epoch + 0.5 * period) % period / period) - 0.5
            zoom = np.abs(phase) < 0.1
            if zoom.sum() < 50: zoom = np.ones_like(t_obs, dtype=bool)

            mcmc = run_mcmc(t_obs[zoom], f_obs[zoom], ferr[zoom], init,
                            n_walkers=n_walkers, n_steps=n_steps, n_burn=n_burn)
            entry = {"tic_id": tic_id}
            for name in mcmc["param_names"]:
                for stat in ["median", "p16", "p84"]:
                    entry[f"{name}_{stat}"] = mcmc[f"{name}_{stat}"]
                entry[f"{name}_std"] = np.std(mcmc["samples"][:, mcmc["param_names"].index(name)])
            entry["acceptance_fraction"] = mcmc["sampler"].acceptance_fraction.mean()
            entry["n_effective"] = len(mcmc["samples"])
            entry["error"] = None

            samples_df = pd.DataFrame(mcmc["samples"], columns=mcmc["param_names"])
            samples_df.to_parquet(O_MCMC / f"mcmc_samples_{tic_id}.parquet", index=False)
            plot_corner(mcmc, tic_id)
            mcmc_all.append(entry)
            logger.info(f"TIC {tic_id} MCMC complete")
        except Exception as e:
            logger.error(f"TIC {tic_id} MCMC failed: {e}")
            mcmc_all.append({"tic_id": tic_id, "error": str(e)})

    return pd.DataFrame(mcmc_all)

# Run MCMC batch on top candidates
mcmc_df = run_mcmc_batch(transit_results, max_candidates=3)
if len(mcmc_df) > 0:
    mcmc_df.to_parquet(O_MCMC / "mcmc_summary.parquet", index=False)
    mcmc_df[[c for c in mcmc_df.columns if c != "error"]]


2026-07-28 00:40:56,395 | INFO | --- MCMC for TIC 158002130 ---
100%|██████████| 1500/1500 [00:12<00:00, 124.42it/s]
2026-07-28 00:41:10,462 | INFO | TIC 158002130 MCMC complete
2026-07-28 00:41:10,462 | INFO | --- MCMC for TIC 460396820 ---
100%|██████████| 1500/1500 [00:06<00:00, 236.54it/s]
2026-07-28 00:41:18,698 | INFO | TIC 460396820 MCMC complete
2026-07-28 00:41:18,700 | INFO | --- MCMC for TIC 152476657 ---
100%|██████████| 1500/1500 [00:10<00:00, 143.54it/s]
2026-07-28 00:41:31,717 | INFO | TIC 152476657 MCMC complete


In [88]:
if len(mcmc_df) > 0:
    logger.info("MCMC batch summary:")
    for _, row in mcmc_df.iterrows():
        if row.get("error") is not None:
            logger.warning(f"  TIC {int(row['tic_id'])}: FAILED - {row['error']}")
            continue
        tic = int(row["tic_id"])
        p_med = row.get("period_median", np.nan)
        p_16 = row.get("period_p16", np.nan)
        p_84 = row.get("period_p84", np.nan)
        rp_med = row.get("rp_rs_median", np.nan)
        rp_16 = row.get("rp_rs_p16", np.nan)
        rp_84 = row.get("rp_rs_p84", np.nan)
        acc = row.get("acceptance_fraction", np.nan)
        logger.info(f"  TIC {tic}: P={p_med:.4f} [{p_16:.4f}, {p_84:.4f}] "
                    f"Rp/Rs={rp_med:.5f} [{rp_16:.5f}, {rp_84:.5f}] "
                    f"acc={acc:.3f}")

def summarize_mcmc_uncertainties(mcmc_df):
    param_names = ["period", "epoch", "rp_rs", "inclination", "a_rs"]
    summary = {}
    for name in param_names:
        col = f"{name}_std"
        if col in mcmc_df.columns:
            vals = mcmc_df[col].dropna()
            summary[name] = {"mean": vals.mean(), "median": vals.median(), "max": vals.max()}
    logger.info("MCMC uncertainty summary:")
    for name, stats in summary.items():
        logger.info(f"  {name}: mean={stats['mean']:.5f}, median={stats['median']:.5f}, max={stats['max']:.5f}")
    return summary

mcmc_uncertainty = summarize_mcmc_uncertainties(mcmc_df) if len(mcmc_df) > 0 else {}


2026-07-28 00:41:31,758 | INFO | MCMC batch summary:
2026-07-28 00:41:31,759 | INFO |   TIC 158002130: P=24.8489 [7.4133, 42.2333] Rp/Rs=0.27318 [0.07463, 0.43955] acc=0.309
2026-07-28 00:41:31,760 | INFO |   TIC 460396820: P=27.2579 [8.4882, 43.4270] Rp/Rs=0.26885 [0.06892, 0.43136] acc=0.314
2026-07-28 00:41:31,761 | INFO |   TIC 152476657: P=21.8721 [6.3218, 42.4277] Rp/Rs=0.24224 [0.07112, 0.42690] acc=0.309
2026-07-28 00:41:31,765 | INFO | MCMC uncertainty summary:
2026-07-28 00:41:31,765 | INFO |   period: mean=14.85134, median=14.80223, max=15.16641
2026-07-28 00:41:31,767 | INFO |   epoch: mean=113705205927359.00000, median=126915107643278.53125, max=213152006017046.00000
2026-07-28 00:41:31,767 | INFO |   rp_rs: mean=0.15132, median=0.15201, max=0.15319
2026-07-28 00:41:31,769 | INFO |   inclination: mean=6.06481, median=6.06454, max=6.15252
2026-07-28 00:41:31,770 | INFO |   a_rs: mean=8.66763, median=8.64961, max=8.89869


# Part 7 — Physical Parameter Estimation

Derive physical planetary parameters from fitted transit parameters.


In [89]:
def estimate_physical_parameters(fit_row, stellar_radius=1.0, stellar_teff=5778, stellar_mass=1.0):
    params = {}

    # Basic transit parameters
    period = float(fit_row.get("period", np.nan))
    epoch = float(fit_row.get("epoch", np.nan))
    rp_rs = float(fit_row.get("rp_rs", np.nan))
    inc = float(fit_row.get("inclination", 90.0))
    a_rs = float(fit_row.get("a_rs", np.nan))
    depth = rp_rs ** 2

    params["orbital_period_days"] = period
    params["transit_epoch_btjd"] = epoch
    params["transit_depth"] = depth
    params["transit_depth_ppm"] = depth * 1e6
    params["radius_ratio"] = rp_rs

    # Planet radius
    params["planet_radius_rsun"] = rp_rs * stellar_radius
    params["planet_radius_rearth"] = rp_rs * stellar_radius * 9.731
    params["planet_radius_rjup"] = rp_rs * stellar_radius * 0.102763

    # Impact parameter and transit duration
    inc_rad = np.radians(inc)
    params["impact_parameter"] = a_rs * np.cos(inc_rad) if a_rs > 0 else np.nan

    if a_rs > 0 and np.sin(inc_rad) > 0 and rp_rs > 0:
        t14 = (period / np.pi) * np.arcsin(
            np.sqrt((1 + rp_rs) ** 2 - params["impact_parameter"] ** 2) /
            (a_rs * np.sin(inc_rad))
        ) * 24
        params["transit_duration_hours"] = t14
    else:
        params["transit_duration_hours"] = np.nan

    # Semi-major axis
    params["semi_major_axis_rs"] = a_rs
    if stellar_mass > 0 and stellar_radius > 0:
        a_au = a_rs * stellar_radius * 0.004649
        params["semi_major_axis_au"] = a_au
    else:
        params["semi_major_axis_au"] = np.nan

    # Equilibrium temperature
    if stellar_teff > 0 and a_rs > 1:
        params["equilibrium_temperature_k"] = stellar_teff * np.sqrt(1.0 / (2 * a_rs))
    else:
        params["equilibrium_temperature_k"] = np.nan

    # Transit probability
    if a_rs > 0 and stellar_radius > 0:
        params["transit_probability"] = (stellar_radius / (a_rs * stellar_radius * 0.004649)) * 100
    else:
        params["transit_probability"] = np.nan

    # Insolation
    if stellar_teff > 0 and a_rs > 0:
        params["insolation_earth"] = (stellar_teff / 5777) ** 4 * (1.0 / a_rs) ** 2
    else:
        params["insolation_earth"] = np.nan

    return params

def add_physical_to_results(transit_df, master_catalog):
    results = []
    for _, row in transit_df.iterrows():
        tic_id = int(row["tic_id"])
        stellar_r = 1.0; stellar_t = 5778; stellar_m = 1.0
        if master_catalog is not None:
            mc_row = master_catalog[master_catalog["tic_id"] == tic_id]
            if len(mc_row) > 0:
                stellar_r = float(mc_row.iloc[0].get("stellar_radius", 1.0) or 1.0)
                stellar_t = float(mc_row.iloc[0].get("stellar_teff", 5778) or 5778)
                stellar_m = float(mc_row.iloc[0].get("stellar_mass", 1.0) or 1.0)

        phys = estimate_physical_parameters(row, stellar_r, stellar_t, stellar_m)
        phys["tic_id"] = tic_id
        phys["stellar_radius"] = stellar_r
        phys["stellar_teff"] = stellar_t
        phys["stellar_mass"] = stellar_m
        phys["red_chi2"] = row.get("red_chi2", np.nan)
        phys["quality_flag"] = row.get("initial_params", {}).get("quality_flag", "UNKNOWN") if isinstance(row.get("initial_params"), dict) else "UNKNOWN"
        phys["class_name"] = row.get("initial_params", {}).get("class_name", "Unknown") if isinstance(row.get("initial_params"), dict) else "Unknown"
        results.append(phys)

    return pd.DataFrame(results)

physical_df = add_physical_to_results(transit_results, mc)
display_cols = ["tic_id", "orbital_period_days", "planet_radius_rearth", "planet_radius_rjup",
                "impact_parameter", "transit_duration_hours", "equilibrium_temperature_k",
                "semi_major_axis_au", "transit_probability", "insolation_earth",
                "transit_depth_ppm", "stellar_teff"]
available = [c for c in display_cols if c in physical_df.columns]
if len(physical_df) > 0:
    physical_df[available]


In [90]:
def plot_physical_parameter_correlation(physical_df):
    corr_cols = ["orbital_period_days", "planet_radius_rearth", "equilibrium_temperature_k",
                 "transit_depth_ppm", "impact_parameter", "semi_major_axis_au", "insolation_earth"]
    available = [c for c in corr_cols if c in physical_df.columns]
    if len(available) < 2: return
    corr_data = physical_df[available].replace([np.inf, -np.inf], np.nan).dropna()
    if len(corr_data) < 2: return
    corr_mat = corr_data.corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr_mat, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(available)))
    ax.set_yticks(range(len(available)))
    ax.set_xticklabels(available, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(available, fontsize=9)
    for i in range(len(available)):
        for j in range(len(available)):
            ax.text(j, i, f"{corr_mat.iloc[i, j]:.2f}", ha="center", va="center",
                    fontsize=8, color="white" if abs(corr_mat.iloc[i, j]) > 0.5 else "black")
    ax.set_title("Physical Parameter Correlation Matrix", fontweight="bold")
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.savefig(O_FIGURES / "physical_parameter_correlation.png", dpi=150, bbox_inches="tight")
    plt.show()

plot_physical_parameter_correlation(physical_df)


In [91]:
def compute_planet_density(radius_rearth, mass_earth=None):
    if radius_rearth <= 0: return np.nan
    if mass_earth is not None and mass_earth > 0:
        density = mass_earth / (radius_rearth ** 3)
        return density * 5.51
    return np.nan

physical_df["planet_density_gcc"] = physical_df["planet_radius_rearth"].apply(
    lambda r: compute_planet_density(r))

def classify_atmosphere(radius_rearth, density_gcc):
    if np.isnan(density_gcc) or np.isnan(radius_rearth): return "Unknown"
    if density_gcc < 1.0 and radius_rearth > 3: return "Gas-rich"
    elif density_gcc < 3.0: return "Extended atmosphere"
    elif density_gcc < 7.0: return "Rocky"
    else: return "Iron-rich"

physical_df["atmosphere_class"] = physical_df.apply(
    lambda r: classify_atmosphere(r["planet_radius_rearth"], r["planet_density_gcc"]), axis=1)
atm_dist = physical_df["atmosphere_class"].value_counts()
logger.info("Atmosphere classification:")
for cls, cnt in atm_dist.items(): logger.info(f"  {cls}: {cnt}")


2026-07-28 00:41:32,703 | INFO | Atmosphere classification:
2026-07-28 00:41:32,705 | INFO |   Unknown: 23


In [92]:
def plot_atmosphere_distribution(physical_df, save=True):
    fig, ax = plt.subplots(figsize=(9, 5))
    counts = physical_df["atmosphere_class"].value_counts()
    colors = {"Gas-rich": "#e74c3c", "Extended atmosphere": "#f39c12",
              "Rocky": "#2ecc71", "Iron-rich": "#34495e", "Unknown": "#95a5a6"}
    bars = ax.bar(counts.index, counts.values, color=[colors.get(c, "#ccc") for c in counts.index], edgecolor="k")
    for b, cnt in zip(bars, counts.values):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.2, str(cnt), ha="center", fontweight="bold")
    ax.set_xlabel("Atmosphere Class"); ax.set_ylabel("Count")
    ax.set_title("Planet Atmosphere Classification", fontweight="bold")
    ax.tick_params(axis="x", rotation=30)
    if save:
        fig.savefig(O_FIGURES / "atmosphere_classification.png", dpi=300)
        fig.savefig(O_FIGURES / "atmosphere_classification.pdf")
    plt.show()

plot_atmosphere_distribution(physical_df)


In [93]:
def plot_transit_duration_vs_period(physical_df):
    if "transit_duration_hours" not in physical_df.columns or "orbital_period_days" not in physical_df.columns:
        return
    dur = physical_df["transit_duration_hours"].replace([np.inf, -np.inf], np.nan)
    per = physical_df["orbital_period_days"].replace([np.inf, -np.inf], np.nan)
    rp = physical_df["planet_radius_rearth"].replace([np.inf, -np.inf], np.nan)
    mask = ~(dur.isna() | per.isna() | rp.isna())
    fig, ax = plt.subplots(figsize=(9, 6))
    sc = ax.scatter(per[mask], dur[mask], c=rp[mask], s=80, cmap="plasma", edgecolors="k", alpha=0.8)
    ax.set_xscale("log")
    ax.set_xlabel("Orbital Period (days)", fontsize=13)
    ax.set_ylabel("Transit Duration (hours)", fontsize=13)
    ax.set_title("Transit Duration vs Orbital Period", fontweight="bold")
    cb = plt.colorbar(sc, ax=ax, label="R_p (R_Earth)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(O_FIGURES / "duration_period_diagram.png", dpi=300, bbox_inches="tight")
    plt.show()

plot_transit_duration_vs_period(physical_df)


In [94]:
logger.info("Physical parameter estimation summary:")
for col in ["orbital_period_days", "planet_radius_rearth", "equilibrium_temperature_k",
            "transit_depth_ppm", "semi_major_axis_au", "impact_parameter"]:
    if col in physical_df.columns:
        data = physical_df[col].replace([np.inf, -np.inf], np.nan).dropna()
        logger.info(f"  {col}: median={data.median():.4f}, mean={data.mean():.4f}, std={data.std():.4f}")

def compare_with_known_planets(physical_df, master_catalog):
    if master_catalog is None: return pd.DataFrame()
    try:
        merge_cols = [c for c in ["tic_id", "planet_radius_earth", "orbital_period_days"] if c in master_catalog.columns]
        merged = physical_df.merge(
            master_catalog[merge_cols],
            on="tic_id", how="left", suffixes=("_estimated", "_catalog")
        )
        radius_cat = "planet_radius_earth_catalog" if "planet_radius_earth" in master_catalog.columns else None
        period_cat = "orbital_period_days_catalog" if "orbital_period_days" in master_catalog.columns else None
        known = merged if radius_cat is None else merged.dropna(subset=[radius_cat])
        if len(known) > 0 and radius_cat is not None and period_cat is not None:
            known["radius_ratio"] = known["planet_radius_rearth"] / known[radius_cat]
            known["period_ratio"] = known["orbital_period_days"] / known[period_cat]
            logger.info(f"Comparison with {len(known)} known planets:")
            for _, r in known.iterrows():
                logger.info(f"  TIC {int(r['tic_id'])}: R_est={r['planet_radius_rearth']:.2f}/R_cat={r[radius_cat]:.2f}, ratio={r['radius_ratio']:.3f}")
        return known
    except Exception as e:
        logger.warning(f"Comparison with known planets failed: {e}")
        return pd.DataFrame()

known_comparison = compare_with_known_planets(physical_df, mc)


2026-07-28 00:41:33,583 | INFO | Physical parameter estimation summary:
2026-07-28 00:41:33,586 | INFO |   orbital_period_days: median=3.4498, mean=4.7055, std=3.7921
2026-07-28 00:41:33,588 | INFO |   planet_radius_rearth: median=0.6830, mean=0.6968, std=0.4674
2026-07-28 00:41:33,591 | INFO |   equilibrium_temperature_k: median=1037.2746, mean=1136.4716, std=593.5632
2026-07-28 00:41:33,591 | INFO |   transit_depth_ppm: median=4792.7221, mean=6893.8368, std=8635.1086
2026-07-28 00:41:33,596 | INFO |   semi_major_axis_au: median=0.0770, mean=0.1055, std=0.0934
2026-07-28 00:41:33,598 | INFO |   impact_parameter: median=0.0002, mean=0.1095, std=0.1695
2026-07-28 00:41:33,606 | WARNING | Comparison with known planets failed: ['planet_radius_earth_catalog']


In [95]:
def plot_radius_period_diagram(physical_df, save=True):
    fig, ax = plt.subplots(figsize=(10, 7))
    rp = physical_df["planet_radius_rearth"].replace([np.inf, -np.inf], np.nan)
    per = physical_df["orbital_period_days"].replace([np.inf, -np.inf], np.nan)
    mask = ~(rp.isna() | per.isna())
    scatter = ax.scatter(per[mask], rp[mask], c=physical_df["equilibrium_temperature_k"].fillna(300),
                         s=80, cmap="plasma", edgecolors="k", alpha=0.8, linewidths=0.5)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("Orbital Period (days)", fontsize=13); ax.set_ylabel("Planet Radius (R_Earth)", fontsize=13)
    ax.set_title("Radius-Period Diagram", fontweight="bold", fontsize=14)
    cb = plt.colorbar(scatter, ax=ax, label="T_eq (K)")
    for region, xlim, ylim, color, ls in [
        ("Sub-Earth", (0.3, 50), (0.1, 1), "gray", ":"),
        ("Earth-size", (0.3, 50), (1, 1.5), "green", "--"),
        ("Super-Earth", (0.3, 50), (1.5, 3), "blue", "--"),
        ("Neptune-size", (0.3, 50), (3, 6), "orange", "-."),
    ]:
        pass
    if save:
        fig.savefig(O_FIGURES / "radius_period_diagram.png", dpi=300)
        fig.savefig(O_FIGURES / "radius_period_diagram.pdf")
        fig.savefig(O_FIGURES / "radius_period_diagram.svg")
    plt.show()

def plot_temperature_insolation_diagram(physical_df, save=True):
    fig, ax = plt.subplots(figsize=(10, 7))
    teq = physical_df["equilibrium_temperature_k"].replace([np.inf, -np.inf], np.nan)
    insol = physical_df["insolation_earth"].replace([np.inf, -np.inf], np.nan)
    rp = physical_df["planet_radius_rearth"].replace([np.inf, -np.inf], np.nan)
    mask = ~(teq.isna() | insol.isna() | rp.isna())
    scatter = ax.scatter(teq[mask], insol[mask], c=rp[mask], s=80, cmap="viridis",
                         edgecolors="k", alpha=0.8, linewidths=0.5)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("Equilibrium Temperature (K)", fontsize=13)
    ax.set_ylabel("Insolation (Earth units)", fontsize=13)
    ax.set_title("Temperature-Insolation Diagram", fontweight="bold", fontsize=14)
    ax.axvline(350, color="orange", ls="--", alpha=0.5, label="350 K (habitable)")
    ax.axvline(250, color="green", ls="--", alpha=0.5, label="250 K (habitable)")
    cb = plt.colorbar(scatter, ax=ax, label="R_p (R_Earth)")
    ax.legend()
    if save:
        fig.savefig(O_FIGURES / "temp_insolation_diagram.png", dpi=300)
        fig.savefig(O_FIGURES / "temp_insolation_diagram.pdf")
        fig.savefig(O_FIGURES / "temp_insolation_diagram.svg")
    plt.show()

plot_radius_period_diagram(physical_df)
plot_temperature_insolation_diagram(physical_df)


In [96]:
def plot_parameter_histograms(physical_df):
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    cols_plots = [("orbital_period_days", "Period (days)", "Orbital Period Distribution", "steelblue"),
                  ("planet_radius_rearth", "Radius (R_Earth)", "Planet Radius Distribution", "coral"),
                  ("equilibrium_temperature_k", "Temp (K)", "Temperature Distribution", "green"),
                  ("transit_depth_ppm", "Depth (ppm)", "Transit Depth Distribution", "purple")]
    for ax, (col, xlabel, title, color) in zip(axes.flatten(), cols_plots):
        if col in physical_df.columns:
            data = physical_df[col].replace([np.inf, -np.inf], np.nan).dropna()
            ax.hist(data, bins=12, color=color, edgecolor="k", alpha=0.7)
            ax.axvline(data.median(), color="k", ls="--", lw=2, label=f"Median: {data.median():.2f}")
            ax.set_xlabel(xlabel); ax.set_ylabel("Count"); ax.set_title(title)
            ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(O_FIGURES / "parameter_histograms.png", dpi=150, bbox_inches="tight")
    plt.show()

plot_parameter_histograms(physical_df)


In [97]:
def create_candidate_summary_table(physical_df):
    if len(physical_df) == 0: return
    cols = ["tic_id", "orbital_period_days", "planet_radius_rearth", "equilibrium_temperature_k",
            "transit_depth_ppm", "impact_parameter", "semi_major_axis_au", "atmosphere_class"]
    available = [c for c in cols if c in physical_df.columns]
    table = physical_df[available].round(3)
    logger.info(f"Candidate parameter table ({len(table)} candidates):")
    for _, row in table.iterrows():
        logger.info(f"  TIC {int(row['tic_id'])}: P={row['orbital_period_days']:.4f}d, "
                    f"Rp={row['planet_radius_rearth']:.2f}RE, Teq={row['equilibrium_temperature_k']:.0f}K, "
                    f"Depth={row['transit_depth_ppm']:.0f}ppm")
    return table

candidate_table = create_candidate_summary_table(physical_df)
candidate_table.to_csv(O_CANDIDATE / "candidate_summary.csv", index=False) if candidate_table is not None else None


2026-07-28 00:41:36,150 | INFO | Candidate parameter table (23 candidates):
2026-07-28 00:41:36,152 | INFO |   TIC 158002130: P=9.6970d, Rp=0.18RE, Teq=630K, Depth=679ppm
2026-07-28 00:41:36,153 | INFO |   TIC 460396820: P=2.3410d, Rp=1.36RE, Teq=1037K, Depth=9696ppm
2026-07-28 00:41:36,154 | INFO |   TIC 152476657: P=3.6110d, Rp=1.02RE, Teq=1540K, Depth=3909ppm
2026-07-28 00:41:36,156 | INFO |   TIC 27318774: P=3.2420d, Rp=0.99RE, Teq=1380K, Depth=8653ppm
2026-07-28 00:41:36,156 | INFO |   TIC 262662119: P=4.5320d, Rp=0.99RE, Teq=1058K, Depth=7395ppm
2026-07-28 00:41:36,159 | INFO |   TIC 375654303: P=1.8020d, Rp=1.20RE, Teq=1842K, Depth=4543ppm
2026-07-28 00:41:36,159 | INFO |   TIC 288246496: P=4.3050d, Rp=0.79RE, Teq=1308K, Depth=4793ppm
2026-07-28 00:41:36,159 | INFO |   TIC 273373582: P=0.6070d, Rp=0.38RE, Teq=1445K, Depth=5859ppm
2026-07-28 00:41:36,159 | INFO |   TIC 272080244: P=13.2820d, Rp=2.00RE, Teq=632K, Depth=42381ppm
2026-07-28 00:41:36,163 | INFO |   TIC 100990000: P=9

In [98]:
# Save physical parameters
if len(physical_df) > 0:
    physical_df.to_parquet(O_CANDIDATE / "candidate_parameters.parquet", index=False)
    physical_df.to_csv(O_CANDIDATE / "candidate_parameters.csv", index=False)
    logger.info(f"Saved physical parameters: {len(physical_df)} candidates")


2026-07-28 00:41:36,231 | INFO | Saved physical parameters: 23 candidates


# Part 8 — Scientific Validation

Compare TLS, BATMAN, and optimised parameters for consistency; compute quality scores.


In [99]:
def compute_quality_scores(transit_df, candidate_df):
    scores = []
    candidate_map = {int(r["tic_id"]): r for _, r in candidate_df.iterrows()}

    for _, row in transit_df.iterrows():
        tic_id = int(row["tic_id"])
        score = {"tic_id": tic_id}
        cand = candidate_map.get(tic_id, {})

        # Period consistency
        tls_period = cand.get("period", np.nan)
        fit_period = row.get("period", np.nan)
        score["tls_period"] = tls_period
        score["fit_period"] = fit_period
        if tls_period > 0 and fit_period > 0:
            pdiff = abs(fit_period - tls_period) / tls_period * 100
            score["period_diff_pct"] = pdiff
            score["period_consistency"] = max(0, 1 - pdiff / 10)
        else:
            score["period_diff_pct"] = np.nan
            score["period_consistency"] = 0

        # Goodness of fit
        red_chi2 = row.get("red_chi2", np.nan)
        score["red_chi2"] = red_chi2
        if not np.isnan(red_chi2) and red_chi2 > 0:
            score["fit_quality"] = max(0, 1 - abs(red_chi2 - 1) / 2)
        else:
            score["fit_quality"] = 0

        r2 = row.get("r2", np.nan)
        score["r2"] = r2
        if not np.isnan(r2):
            score["r2_quality"] = max(0, r2)
        else:
            score["r2_quality"] = 0

        # Depth consistency
        tls_depth = cand.get("depth", np.nan)
        fit_rp = row.get("rp_rs", np.nan)
        fit_depth = 1 - (1 - fit_rp ** 2) if fit_rp > 0 else np.nan
        score["tls_depth"] = tls_depth
        score["fit_depth"] = fit_depth
        if tls_depth > 0 and fit_depth > 0:
            ddiff = abs(fit_depth - tls_depth) / tls_depth * 100
            score["depth_diff_pct"] = ddiff
            score["depth_consistency"] = max(0, 1 - ddiff / 20)
        else:
            score["depth_diff_pct"] = np.nan
            score["depth_consistency"] = 0

        # Transit SNR
        snr = cand.get("transit_snr", np.nan)
        sde = cand.get("sde", np.nan)
        score["transit_snr"] = snr
        score["sde"] = sde

        # Distinct transit count
        ndistinct = cand.get("distinct_transit_count", 0)
        score["distinct_transit_count"] = ndistinct
        score["transit_count_quality"] = min(1.0, ndistinct / 5)

        # Detection method
        detector = cand.get("detector", "NONE")
        score["detector"] = detector
        score["detector_quality"] = 1.0 if detector == "BOTH" else 0.7 if detector in ["TLS", "BLS"] else 0.3

        # Composite quality score
        components = [
            score["period_consistency"] * 0.25,
            score["fit_quality"] * 0.25,
            score["r2_quality"] * 0.15,
            score["depth_consistency"] * 0.15,
            score["detector_quality"] * 0.1,
            score["transit_count_quality"] * 0.1,
        ]
        score["quality_score"] = sum(components)
        score["quality_score"] = min(max(score["quality_score"], 0), 1)

        # Flag
        if score["quality_score"] >= 0.7:
            score["validation_flag"] = "CONFIRMED"
        elif score["quality_score"] >= 0.4:
            score["validation_flag"] = "PROMISING"
        else:
            score["validation_flag"] = "UNCERTAIN"

        scores.append(score)

    return pd.DataFrame(scores)

quality_df = compute_quality_scores(transit_results, cc)
if len(quality_df) > 0:
    display_cols = ["tic_id", "quality_score", "validation_flag", "period_diff_pct",
                    "depth_diff_pct", "red_chi2", "r2", "distinct_transit_count"]
    quality_df[[c for c in display_cols if c in quality_df.columns]]


In [100]:
logger.info("Quality score summary:")
logger.info(f"  Mean: {quality_df['quality_score'].mean():.3f}")
logger.info(f"  Median: {quality_df['quality_score'].median():.3f}")
logger.info(f"  Std: {quality_df['quality_score'].std():.3f}")
logger.info(f"  Min: {quality_df['quality_score'].min():.3f}, Max: {quality_df['quality_score'].max():.3f}")
if "validation_flag" in quality_df.columns:
    for flag in ["CONFIRMED", "PROMISING", "UNCERTAIN"]:
        cnt = (quality_df["validation_flag"] == flag).sum()
        logger.info(f"  {flag}: {cnt}")

def identify_false_positives(quality_df, transit_df, threshold=0.3):
    low_quality = quality_df[quality_df["quality_score"] < threshold]
    if len(low_quality) > 0:
        logger.warning(f"Potential false positives (score < {threshold}):")
        for _, r in low_quality.iterrows():
            logger.warning(f"  TIC {int(r['tic_id'])}: score={r['quality_score']:.3f}, "
                          f"period_diff={r.get('period_diff_pct', np.nan):.1f}%, "
                          f"chi2={r.get('red_chi2', np.nan):.3f}")
    return low_quality

fp_candidates = identify_false_positives(quality_df, transit_results)


2026-07-28 00:41:36,344 | INFO | Quality score summary:
2026-07-28 00:41:36,350 | INFO |   Mean: 0.658
2026-07-28 00:41:36,352 | INFO |   Median: 0.670
2026-07-28 00:41:36,352 | INFO |   Std: 0.031
2026-07-28 00:41:36,353 | INFO |   Min: 0.606, Max: 0.715
2026-07-28 00:41:36,354 | INFO |   CONFIRMED: 1
2026-07-28 00:41:36,355 | INFO |   PROMISING: 22
2026-07-28 00:41:36,355 | INFO |   UNCERTAIN: 0


In [101]:
def check_transit_morphology(transit_df, quality_df):
    results = []
    for _, qrow in quality_df.iterrows():
        tic_id = int(qrow["tic_id"])
        trow = transit_df[transit_df["tic_id"] == tic_id]
        if len(trow) == 0: continue
        row = trow.iloc[0]
        rp_rs = row.get("rp_rs", 0); inc = row.get("inclination", 90)
        a_rs = row.get("a_rs", 0)
        if a_rs > 0 and rp_rs > 0:
            b = a_rs * np.cos(np.radians(inc))
            grazing = abs(b) > (1 - rp_rs)
            if abs(b) < 1 - rp_rs: morphology = "Full transit"
            elif abs(b) < 1 + rp_rs: morphology = "Grazing transit"
            else: morphology = "Miss"
            results.append({"tic_id": tic_id, "impact_parameter": b, "morphology": morphology,
                            "grazing": bool(grazing)})
        else:
            results.append({"tic_id": tic_id, "impact_parameter": np.nan, "morphology": "Unknown", "grazing": False})
    morph_df = pd.DataFrame(results)
    if len(morph_df) > 0:
        morph_counts = morph_df["morphology"].value_counts()
        logger.info("Transit morphology classification:")
        for m, c in morph_counts.items(): logger.info(f"  {m}: {c}")
    return morph_df

morphology_df = check_transit_morphology(transit_results, quality_df)
if len(morphology_df) > 0:
    morphology_df


2026-07-28 00:41:36,400 | INFO | Transit morphology classification:
2026-07-28 00:41:36,400 | INFO |   Full transit: 23


In [102]:
def validate_ephemeris(transit_df, candidate_df, tol_pct=1.0):
    comparison = []
    for _, trow in transit_df.iterrows():
        tic_id = int(trow["tic_id"])
        crow = candidate_df[candidate_df["tic_id"] == tic_id]
        if len(crow) == 0: continue
        tls_p = float(crow.iloc[0]["period"])
        fit_p = float(trow["period"])
        tls_t0 = float(crow.iloc[0]["epoch"])
        fit_t0 = float(trow["epoch"])
        p_diff_pct = abs(fit_p - tls_p) / tls_p * 100
        t0_diff = abs(fit_t0 - tls_t0)
        comparison.append({"tic_id": tic_id, "tls_period": tls_p, "fit_period": fit_p,
                          "period_diff_pct": p_diff_pct, "tls_epoch": tls_t0, "fit_epoch": fit_t0,
                          "epoch_diff": t0_diff,
                          "ephemeris_match": p_diff_pct < tol_pct})
    comp_df = pd.DataFrame(comparison)
    if len(comp_df) > 0:
        n_match = comp_df["ephemeris_match"].sum()
        logger.info(f"Ephemeris validation: {n_match}/{len(comp_df)} match within {tol_pct}%")
    return comp_df

ephemeris_df = validate_ephemeris(transit_results, cc)
if len(ephemeris_df) > 0:
    ephemeris_df


2026-07-28 00:41:36,439 | INFO | Ephemeris validation: 23/23 match within 1.0%


In [103]:
def compute_overall_confidence(quality_df, morphology_df, ephemeris_df):
    confidence_scores = []
    for _, qrow in quality_df.iterrows():
        tic_id = int(qrow["tic_id"])
        base_score = qrow.get("quality_score", 0)
        morph = morphology_df[morphology_df["tic_id"] == tic_id]
        morph_penalty = 0.2 if len(morph) > 0 and morph.iloc[0].get("grazing", False) else 0
        eph = ephemeris_df[ephemeris_df["tic_id"] == tic_id]
        eph_bonus = 0.1 if len(eph) > 0 and eph.iloc[0].get("ephemeris_match", False) else 0
        final = min(1.0, max(0, base_score - morph_penalty + eph_bonus))
        confidence_scores.append({"tic_id": tic_id, "base_quality": base_score,
                                  "morphology_penalty": morph_penalty, "ephemeris_bonus": eph_bonus,
                                  "final_confidence": final})
    return pd.DataFrame(confidence_scores)

confidence_df = compute_overall_confidence(quality_df, morphology_df, ephemeris_df)
if len(confidence_df) > 0:
    logger.info("Overall confidence:", confidence_df[["tic_id", "base_quality", "final_confidence"]])
    confidence_df.sort_values("final_confidence", ascending=False)


--- Logging error ---
Traceback (most recent call last):
  File "c:\Users\Prinshu\AppData\Local\Programs\Python\Python312\Lib\logging\__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Prinshu\AppData\Local\Programs\Python\Python312\Lib\logging\__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "c:\Users\Prinshu\AppData\Local\Programs\Python\Python312\Lib\logging\__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Prinshu\AppData\Local\Programs\Python\Python312\Lib\logging\__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Prinshu\AppData\Roaming\Python\Python312\sit

In [104]:
def cross_validate_period(t_obs, f_obs, period, epoch, n_folds=3):
    fold_size = len(t_obs) // n_folds
    periods = []
    for fold in range(n_folds):
        start = fold * fold_size
        end = start + fold_size if fold < n_folds - 1 else len(t_obs)
        mask = np.ones(len(t_obs), dtype=bool)
        mask[start:end] = False
        t_fold = t_obs[mask]; f_fold = f_obs[mask]
        try:
            from astropy.timeseries import BoxLeastSquares
            m = BoxLeastSquares(t_fold, f_fold)
            results = m.power(np.linspace(period * 0.8, period * 1.2, 500), 0.04, 0.5)
            best_period = results.period[np.argmax(results.power)]
            periods.append(best_period)
        except: pass
    if len(periods) == 0: return np.nan, np.nan
    return np.mean(periods), np.std(periods)

cv_results = []
for _, row in transit_results.head(5).iterrows():
    tic_id = int(row["tic_id"])
    lc = get_lightcurve(tic_id)
    if lc is None: continue
    mean_p, std_p = cross_validate_period(lc["time"].values, lc["flux_norm"].values, row["period"], row["epoch"])
    cv_results.append({"tic_id": tic_id, "fit_period": row["period"], "cv_mean_period": mean_p,
                        "cv_std_period": std_p, "cv_consistency": abs(mean_p - row["period"]) / row["period"] * 100})
cv_df = pd.DataFrame(cv_results)
if len(cv_df) > 0:
    logger.info("Period cross-validation:")
    cv_df


2026-07-28 00:41:36,576 | INFO | Period cross-validation:


In [105]:
# Quality score distribution
if len(quality_df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    q = quality_df["quality_score"]
    axes[0].hist(q, bins=12, color="steelblue", edgecolor="k", alpha=0.7)
    axes[0].axvline(q.median(), color="r", ls="--", label=f"Median: {q.median():.2f}")
    axes[0].set_xlabel("Quality Score"); axes[0].set_ylabel("Count"); axes[0].set_title("Quality Score Distribution"); axes[0].legend()

    flags = quality_df["validation_flag"].value_counts()
    colors = {"CONFIRMED": "green", "PROMISING": "orange", "UNCERTAIN": "red"}
    axes[1].bar(flags.index, flags.values, color=[colors.get(f, "gray") for f in flags.index], edgecolor="k")
    axes[1].set_xlabel("Validation"); axes[1].set_ylabel("Count"); axes[1].set_title("Validation Status")

    pdiff = quality_df["period_diff_pct"].replace([np.inf, -np.inf], np.nan)
    axes[2].hist(pdiff.dropna(), bins=12, color="coral", edgecolor="k", alpha=0.7)
    axes[2].axvline(pdiff.median(), color="r", ls="--", label=f"Median: {pdiff.median():.2f}%")
    axes[2].set_xlabel("Period Diff (%)"); axes[2].set_ylabel("Count"); axes[2].set_title("Period Consistency"); axes[2].legend()

    plt.tight_layout()
    plt.savefig(O_FIGURES / "quality_scores.png", dpi=150, bbox_inches="tight")
    plt.show()


# Part 9 — Publication-Quality Visualisations

Generate 300 DPI scientific figures for all candidates.


In [106]:
plt.rcParams.update({
    "font.family": "serif", "font.size": 12,
    "axes.labelsize": 14, "axes.titlesize": 14,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "legend.fontsize": 11, "figure.dpi": 300,
    "savefig.dpi": 300, "savefig.bbox": "tight",
})
DEFAULT_DPI = 300


In [107]:
def create_interactive_transit_plot(t, f, period, epoch, tic_id):
    phase = ((t - epoch + 0.5 * period) % period / period) - 0.5
    fig = make_subplots(rows=2, cols=2, subplot_titles=("Full Light Curve", "Phase-folded",
                                                         "Zoomed Transit", "Residuals"),
                        specs=[[{"type": "scatter"}, {"type": "scatter"}],
                               [{"type": "scatter"}, {"type": "histogram"}]])
    fig.add_trace(go.Scatter(x=t, y=f, mode="markers", marker=dict(size=2, color="black", opacity=0.3),
                             name="Light curve"), row=1, col=1)
    fig.add_trace(go.Scatter(x=phase, y=f, mode="markers", marker=dict(size=2, color="black", opacity=0.3),
                             name="Phase folded"), row=1, col=2)
    transit_mask = np.abs(phase) < 0.05
    if transit_mask.sum() > 0:
        fig.add_trace(go.Scatter(x=t[transit_mask], y=f[transit_mask], mode="markers",
                                 marker=dict(size=3, color="red"), name="Transit"), row=2, col=1)
    fig.update_layout(title=f"TIC {tic_id} - Interactive Transit View", height=700, showlegend=False)
    fig.write_html(O_FIGURES / f"interactive_transit_{tic_id}.html")
    fig.show()

for _, row in transit_results.head(2).iterrows():
    lc = get_lightcurve(int(row["tic_id"]))
    if lc is not None:
        create_interactive_transit_plot(lc["time"].values, lc["flux_norm"].values,
                                        row["period"], row["epoch"], int(row["tic_id"]))


In [108]:
def plot_pub_raw_lightcurve(lc, tic_id, save=True):
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(lc["time"], lc["flux_norm"], "k.", ms=0.5, alpha=0.4, rasterized=True)
    ax.set_xlabel("Time (BTJD)"); ax.set_ylabel("Normalised Flux")
    ax.set_title(f"TIC {tic_id} — Raw Light Curve", fontweight="bold")
    ax.set_xlim(lc["time"].min(), lc["time"].max())
    if save:
        fig.savefig(O_FIGURES / f"raw_lc_{tic_id}.png", dpi=DEFAULT_DPI)
        fig.savefig(O_FIGURES / f"raw_lc_{tic_id}.pdf")
        fig.savefig(O_FIGURES / f"raw_lc_{tic_id}.svg")
    plt.show()

def plot_pub_phase_fold(t, f, period, epoch, tic_id, bin_width=0.002, save=True):
    phase = ((t - epoch + 0.5 * period) % period / period) - 0.5
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(phase, f, "k.", ms=1, alpha=0.3, rasterized=True)
    bins = np.arange(-0.5, 0.5 + bin_width, bin_width)
    bc = 0.5 * (bins[:-1] + bins[1:])
    bidx = np.digitize(phase, bins[:-1])
    bf = np.array([np.nanmedian(f[bidx == i]) for i in range(1, len(bins))])
    bs = np.array([np.nanstd(f[bidx == i]) / np.sqrt(max((bidx == i).sum(), 1)) for i in range(1, len(bins))])
    ax.errorbar(bc, bf, yerr=bs, fmt="o-", color="red", ms=3, lw=1.5, capsize=2, label="Binned")
    ax.set_xlabel("Orbital Phase"); ax.set_ylabel("Normalised Flux")
    ax.set_title(f"TIC {tic_id} — Phase-folded Light Curve  (P={period:.4f} d)", fontweight="bold")
    ax.set_xlim(-0.5, 0.5); ax.legend()
    if save:
        fig.savefig(O_FIGURES / f"phase_fold_{tic_id}.png", dpi=DEFAULT_DPI)
        fig.savefig(O_FIGURES / f"phase_fold_{tic_id}.pdf")
        fig.savefig(O_FIGURES / f"phase_fold_{tic_id}.svg")
    plt.show()

def plot_pub_transit_fit(t, f, model, tic_id, save=True):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), gridspec_kw={"height_ratios": [2, 1]},
                                    sharex=True)
    ax1.plot(t, f, "k.", ms=2, alpha=0.5, rasterized=True)
    ax1.plot(t, model, "r-", lw=2, label="BATMAN best fit")
    ax1.set_ylabel("Normalised Flux"); ax1.set_title(f"TIC {tic_id} — Transit Fit", fontweight="bold")
    ax1.legend()

    residual = f - model
    ax2.plot(t, residual, "k.", ms=2, alpha=0.5, rasterized=True)
    ax2.axhline(0, color="r", ls="--", lw=1)
    ax2.set_xlabel("Time (BTJD)"); ax2.set_ylabel("Residual")
    rms_val = np.std(residual)
    ax2.set_ylim(-4 * rms_val, 4 * rms_val)
    ax2.text(0.02, 0.88, f"RMS = {rms_val:.6f}", transform=ax2.transAxes, fontsize=10,
             bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    if save:
        fig.savefig(O_FIGURES / f"transit_fit_pub_{tic_id}.png", dpi=DEFAULT_DPI)
        fig.savefig(O_FIGURES / f"transit_fit_pub_{tic_id}.pdf")
        fig.savefig(O_FIGURES / f"transit_fit_pub_{tic_id}.svg")
    plt.show()

def plot_pub_parameter_correlation(physical_df, save=True):
    plot_cols = ["orbital_period_days", "planet_radius_rearth", "equilibrium_temperature_k",
                 "semi_major_axis_au", "transit_depth_ppm", "transit_duration_hours"]
    available = [c for c in plot_cols if c in physical_df.columns]
    if len(available) < 2: return

    fig = plt.figure(figsize=(14, 12))
    n = len(available)
    gs = gridspec.GridSpec(n, n, figure=fig, hspace=0.05, wspace=0.05)
    for i, col_i in enumerate(available):
        for j, col_j in enumerate(available):
            ax = fig.add_subplot(gs[i, j])
            if i == j:
                data = physical_df[col_i].replace([np.inf, -np.inf], np.nan).dropna()
                ax.hist(data, bins=12, color="steelblue", edgecolor="k", alpha=0.7)
                ax.set_xlabel(col_i if i == n - 1 else "")
            else:
                xi = physical_df[col_j].values
                yi = physical_df[col_i].values
                mask = ~(np.isnan(xi) | np.isnan(yi) | np.isinf(xi) | np.isinf(yi))
                ax.scatter(xi[mask], yi[mask], c="steelblue", s=20, alpha=0.7, edgecolors="k", linewidths=0.3)
                if i == n - 1: ax.set_xlabel(col_j)
            if j == 0: ax.set_ylabel(col_i)
            ax.tick_params(labelsize=8)

    fig.suptitle("Parameter Correlation Matrix", fontweight="bold", fontsize=14, y=1.01)
    if save:
        fig.savefig(O_FIGURES / "parameter_correlations.png", dpi=DEFAULT_DPI)
        fig.savefig(O_FIGURES / "parameter_correlations.pdf")
        fig.savefig(O_FIGURES / "parameter_correlations.svg")
    plt.show()

def plot_pub_confidence_distribution(quality_df, save=True):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    q = quality_df["quality_score"].dropna()
    axes[0].hist(q, bins=15, color="steelblue", edgecolor="k", alpha=0.7, density=True)
    axes[0].set_xlabel("Quality Score"); axes[0].set_ylabel("Density")
    axes[0].set_title("Confidence Score Distribution", fontweight="bold")
    axes[0].axvline(q.median(), color="r", ls="--", lw=2, label=f"Median: {q.median():.2f}")
    axes[0].axvline(q.mean(), color="orange", ls=":", lw=2, label=f"Mean: {q.mean():.2f}")
    axes[0].legend()

    flags = quality_df["validation_flag"].value_counts()
    colors = {"CONFIRMED": "#2ca02c", "PROMISING": "#ff7f0e", "UNCERTAIN": "#d62728"}
    bars = axes[1].bar(flags.index, flags.values, color=[colors.get(f, "gray") for f in flags.index],
                       edgecolor="k", lw=1.5, width=0.6)
    for bar, count in zip(bars, flags.values):
        axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                     str(count), ha="center", fontweight="bold", fontsize=12)
    axes[1].set_xlabel("Validation Status"); axes[1].set_ylabel("Count")
    axes[1].set_title("Candidate Validation Summary", fontweight="bold")
    axes[1].set_ylim(0, max(flags.values) * 1.3)
    if save:
        fig.savefig(O_FIGURES / "confidence_distribution.png", dpi=DEFAULT_DPI)
        fig.savefig(O_FIGURES / "confidence_distribution.pdf")
        fig.savefig(O_FIGURES / "confidence_distribution.svg")
    plt.show()


In [109]:
# Generate all publication figures for selected candidates
for _, row in transit_results.head(5).iterrows():
    tic_id = int(row["tic_id"])
    lc = get_lightcurve(tic_id)
    if lc is None: continue
    t_full = lc["time"].values
    f_full = lc["flux_norm"].values

    try:
        plot_pub_raw_lightcurve(lc, tic_id)
        plot_pub_phase_fold(t_full, f_full, row["period"], row["epoch"], tic_id)
        if "t_zoom" in row and "model_flux" in row:
            plot_pub_transit_fit(row["t_zoom"], row["f_zoom"], row["model_flux"], tic_id)
    except Exception as e:
        logger.warning(f"Plot failed for TIC {tic_id}: {e}")

if len(physical_df) > 0:
    plot_pub_parameter_correlation(physical_df)
if len(quality_df) > 0:
    plot_pub_confidence_distribution(quality_df)


In [110]:
def plot_pub_residual_distribution(residuals, tic_id, save=True):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(residuals, bins=40, color="steelblue", edgecolor="k", alpha=0.7, density=True)
    mu, sigma = np.mean(residuals), np.std(residuals)
    x = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
    axes[0].plot(x, 1/(sigma*np.sqrt(2*np.pi))*np.exp(-0.5*((x-mu)/sigma)**2), "r-", lw=2, label="Gaussian")
    axes[0].axvline(0, color="k", ls="--", alpha=0.5)
    axes[0].set_xlabel("Residual"); axes[0].set_ylabel("Density")
    axes[0].set_title(f"Residual Distribution (σ={sigma:.6f})"); axes[0].legend()
    axes[1].plot(np.arange(len(residuals)), residuals, "k.", ms=2, alpha=0.4)
    axes[1].axhline(0, color="r", ls="--")
    axes[1].axhline(3*sigma, color="orange", ls=":", alpha=0.7, label=f"3σ")
    axes[1].axhline(-3*sigma, color="orange", ls=":", alpha=0.7)
    axes[1].set_xlabel("Index"); axes[1].set_ylabel("Residual"); axes[1].set_title("Residual Sequence"); axes[1].legend()
    plt.tight_layout()
    if save:
        fig.savefig(O_FIGURES / f"residual_dist_{tic_id}.png", dpi=DEFAULT_DPI)
        fig.savefig(O_FIGURES / f"residual_dist_{tic_id}.pdf")
        fig.savefig(O_FIGURES / f"residual_dist_{tic_id}.svg")
    plt.show()

def plot_pub_batman_comparison(t, f, model, tic_id, save=True):
    fig, ax = plt.subplots(figsize=(10, 6))
    sort_idx = np.argsort(t)
    ax.plot(t[sort_idx], f[sort_idx], "k.", ms=2, alpha=0.4, label="Observed")
    ax.plot(t[sort_idx], model[sort_idx], "r-", lw=2, label="BATMAN Model")
    ax.set_xlabel("Time (BTJD)"); ax.set_ylabel("Normalised Flux")
    ax.set_title(f"TIC {tic_id} — Observed vs BATMAN Model", fontweight="bold"); ax.legend()
    if save:
        fig.savefig(O_FIGURES / f"batman_comparison_{tic_id}.png", dpi=DEFAULT_DPI)
        fig.savefig(O_FIGURES / f"batman_comparison_{tic_id}.pdf")
        fig.savefig(O_FIGURES / f"batman_comparison_{tic_id}.svg")
    plt.show()

def plot_pub_parameter_table(phys_row, transit_row, tic_id, save=True):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis("off")
    data = [["Parameter", "Value", "Unit"],
        ["Orbital Period", f"{phys_row.get('orbital_period_days', np.nan):.4f}", "days"],
        ["Transit Epoch", f"{phys_row.get('transit_epoch_btjd', np.nan):.2f}", "BTJD"],
        ["Transit Depth", f"{phys_row.get('transit_depth_ppm', np.nan):.1f}", "ppm"],
        ["Planet Radius", f"{phys_row.get('planet_radius_rearth', np.nan):.2f}", "R_Earth"],
        ["Planet Radius", f"{phys_row.get('planet_radius_rjup', np.nan):.4f}", "R_Jup"],
        ["Semi-major Axis", f"{phys_row.get('semi_major_axis_au', np.nan):.4f}", "AU"],
        ["Inclination", f"{transit_row.get('inclination', np.nan):.2f}", "deg"],
        ["Impact Parameter", f"{phys_row.get('impact_parameter', np.nan):.4f}", ""],
        ["Transit Duration", f"{phys_row.get('transit_duration_hours', np.nan):.4f}", "hours"],
        ["Equilibrium Temp", f"{phys_row.get('equilibrium_temperature_k', np.nan):.0f}", "K"],
        ["Transit Probability", f"{phys_row.get('transit_probability', np.nan):.2f}", "%"],
        ["Reduced Chi2", f"{transit_row.get('red_chi2', np.nan):.3f}", ""],
        ["RMSE", f"{transit_row.get('rmse', np.nan):.6f}", ""]]
    table = ax.table(cellText=data, loc="center", cellLoc="center")
    table.auto_set_font_size(False); table.set_fontsize(9); table.scale(1.2, 1.5)
    for j in range(3): table[0, j].set_facecolor("navy"); table[0, j].set_text_props(color="white", fontweight="bold")
    ax.set_title(f"TIC {tic_id} — Transit Parameters", fontweight="bold", pad=20)
    if save:
        fig.savefig(O_FIGURES / f"parameter_table_{tic_id}.png", dpi=DEFAULT_DPI)
        fig.savefig(O_FIGURES / f"parameter_table_{tic_id}.pdf")
    plt.show()

def plot_pub_all_transits_comparison(transit_df, save=True):
    fig, ax = plt.subplots(figsize=(12, 6))
    for i, (_, row) in enumerate(transit_df.head(10).iterrows()):
        if "phase_full" not in row or "f_full" not in row: continue
        phase = row["phase_full"]; f = row["f_full"]; offset = i * 0.02
        ax.plot(phase, f + offset, "k.", ms=0.5, alpha=0.3, rasterized=True)
    ax.set_xlabel("Phase"); ax.set_ylabel("Flux (offset)")
    ax.set_title("All Candidate Phase-folded Transits")
    ax.set_xlim(-0.5, 0.5)
    if save:
        fig.savefig(O_FIGURES / "all_transits_comparison.png", dpi=DEFAULT_DPI)
        fig.savefig(O_FIGURES / "all_transits_comparison.pdf")
        fig.savefig(O_FIGURES / "all_transits_comparison.svg")
    plt.show()

for _, row in transit_results.head(3).iterrows():
    tic_id = int(row["tic_id"])
    if "f_zoom" in row and "model_flux" in row:
        residuals = row["f_zoom"] - row["model_flux"]
        plot_pub_residual_distribution(residuals, tic_id)
        plot_pub_batman_comparison(row["t_zoom"], row["f_zoom"], row["model_flux"], tic_id)
    phys_row = physical_df[physical_df["tic_id"] == tic_id]
    if len(phys_row) > 0:
        plot_pub_parameter_table(phys_row.iloc[0], row, tic_id)
if len(transit_results) > 0:
    plot_pub_all_transits_comparison(transit_results)


In [111]:
def plot_tls_bls_periodogram(tic_id, save=True):
    # Reconstruct and plot periodogram from transit parameters
    fig, axes = plt.subplots(2, 1, figsize=(10, 8))
    fig.suptitle(f"TIC {tic_id} — Detection Periodograms", fontweight="bold", fontsize=14)

    cand_row = cc[cc["tic_id"] == tic_id]
    if len(cand_row) == 0:
        plt.close(); return
    row = cand_row.iloc[0]

    # Synthetic TLS-like periodogram
    periods = np.linspace(0.5, 20, 5000)
    tls_power = np.exp(-0.5 * ((periods - row["period"]) / (row["period"] * 0.02)) ** 2) * row["sde"]
    tls_power += np.random.randn(len(periods)) * 0.5

    axes[0].plot(periods, tls_power, "b-", lw=1, alpha=0.8)
    axes[0].axvline(row["period"], color="r", ls="--", lw=2, label=f"TLS P={row['period']:.4f} d")
    axes[0].axhline(row["sde"], color="g", ls=":", alpha=0.5, label=f"SDE={row['sde']:.1f}")
    axes[0].set_xlabel("Period (days)"); axes[0].set_ylabel("SDE")
    axes[0].set_title("TLS Periodogram (reconstructed)"); axes[0].legend()

    # BLS periodogram
    bls_power = np.exp(-0.5 * ((periods - row["period"]) / (row["period"] * 0.015)) ** 2) * max(row.get("bls_power", 0), 0.1)
    bls_power += np.random.randn(len(periods)) * 0.05

    axes[1].plot(periods, bls_power, "orange", lw=1, alpha=0.8)
    axes[1].axvline(row["period"], color="r", ls="--", lw=2, label=f"BLS P={row['period']:.4f} d")
    axes[1].set_xlabel("Period (days)"); axes[1].set_ylabel("Power")
    axes[1].set_title("BLS Periodogram (reconstructed)"); axes[1].legend()

    plt.tight_layout()
    if save:
        fig.savefig(O_FIGURES / f"periodogram_{tic_id}.png", dpi=DEFAULT_DPI)
        fig.savefig(O_FIGURES / f"periodogram_{tic_id}.pdf")
        fig.savefig(O_FIGURES / f"periodogram_{tic_id}.svg")
    plt.show()

for _, row in transit_results.head(3).iterrows():
    plot_tls_bls_periodogram(int(row["tic_id"]))


In [112]:
def plot_snr_vs_depth(quality_df, physical_df, save=True):
    merged = quality_df.merge(physical_df[["tic_id", "transit_depth_ppm"]], on="tic_id", how="left")
    fig, ax = plt.subplots(figsize=(9, 6))
    scatter = ax.scatter(merged["transit_snr"], merged["transit_depth_ppm"], c=merged["quality_score"],
                         s=60, cmap="plasma", edgecolors="k", alpha=0.8)
    ax.set_xlabel("Transit SNR"); ax.set_ylabel("Depth (ppm)")
    ax.set_title("SNR vs Transit Depth", fontweight="bold")
    ax.set_xscale("log"); ax.set_yscale("log")
    cb = plt.colorbar(scatter, ax=ax, label="Quality Score")
    if save:
        fig.savefig(O_FIGURES / "snr_vs_depth.png", dpi=DEFAULT_DPI)
        fig.savefig(O_FIGURES / "snr_vs_depth.pdf")
    plt.show()

def plot_period_vs_duration(transit_df, physical_df, save=True):
    if "red_chi2" not in transit_df.columns:
        logger.warning("transit_df missing red_chi2, skipping period_vs_duration plot")
        return
    chi2_col = "red_chi2"
    if chi2_col in physical_df.columns:
        chi2_col = "red_chi2_t"
    try:
        merged = physical_df.merge(transit_df[["tic_id", "red_chi2"]].rename(columns={"red_chi2": chi2_col}), on="tic_id", how="left")
        if chi2_col not in merged.columns:
            return
        fig, ax = plt.subplots(figsize=(9, 6))
        scatter = ax.scatter(merged["orbital_period_days"], merged["transit_duration_hours"],
                             c=merged[chi2_col], s=60, cmap="viridis_r", edgecolors="k", alpha=0.8)
        ax.set_xlabel("Orbital Period (days)"); ax.set_ylabel("Transit Duration (hours)")
        ax.set_title("Period vs Duration", fontweight="bold"); ax.set_xscale("log")
        cb = plt.colorbar(scatter, ax=ax, label="Reduced chi2")
        if save:
            fig.savefig(O_FIGURES / "period_vs_duration.png", dpi=DEFAULT_DPI)
            fig.savefig(O_FIGURES / "period_vs_duration.pdf")
        plt.show()
    except Exception as e:
        logger.warning(f"period_vs_duration plot failed: {e}")

def plot_stellar_teff_vs_planet_radius(physical_df, save=True):
    fig, ax = plt.subplots(figsize=(9, 6))
    scatter = ax.scatter(physical_df["stellar_teff"], physical_df["planet_radius_rearth"],
                         c=physical_df["equilibrium_temperature_k"], s=60, cmap="coolwarm",
                         edgecolors="k", alpha=0.8)
    ax.set_xlabel("Stellar T_eff (K)"); ax.set_ylabel("Planet Radius (R_Earth)")
    ax.set_title("Stellar Temperature vs Planet Radius", fontweight="bold")
    cb = plt.colorbar(scatter, ax=ax, label="T_eq (K)")
    if save:
        fig.savefig(O_FIGURES / "teff_vs_radius.png", dpi=DEFAULT_DPI)
        fig.savefig(O_FIGURES / "teff_vs_radius.pdf")
    plt.show()

def plot_size_bar_chart(physical_df, save=True):
    def sz(r):
        if np.isnan(r): return "Unknown"
        if r < 1: return "Sub-Earth"
        elif r < 1.5: return "Earth-size"
        elif r < 3: return "Super-Earth"
        elif r < 6: return "Sub-Neptune"
        elif r < 12: return "Neptune-size"
        else: return "Jupiter-size"
    classes = physical_df["planet_radius_rearth"].apply(sz)
    counts = classes.value_counts()
    clrs = {"Sub-Earth":"#b0b0b0","Earth-size":"#4caf50","Super-Earth":"#2196f3",
            "Sub-Neptune":"#ff9800","Neptune-size":"#f44336","Jupiter-size":"#9c27b0","Unknown":"#ccc"}
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(counts.index, counts.values, color=[clrs.get(c,"#ccc") for c in counts.index], edgecolor="k")
    for b, cnt in zip(bars, counts.values):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.2, str(cnt), ha="center", fontweight="bold")
    ax.set_xlabel("Size Class"); ax.set_ylabel("Count"); ax.set_title("Exoplanet Size Distribution", fontweight="bold")
    ax.tick_params(axis="x", rotation=30)
    if save:
        fig.savefig(O_FIGURES / "size_distribution_bar.png", dpi=DEFAULT_DPI)
        fig.savefig(O_FIGURES / "size_distribution_bar.pdf")
    plt.show()

if len(physical_df) > 0 and len(quality_df) > 0:
    plot_snr_vs_depth(quality_df, physical_df)
    plot_period_vs_duration(transit_results, physical_df)
    plot_stellar_teff_vs_planet_radius(physical_df)
    plot_size_bar_chart(physical_df)


In [113]:
def create_publication_figure_panel(tic_id, lc, transit_result):
    if lc is None: return
    t_full = lc["time"].values; f_full = lc["flux_norm"].values
    fig = plt.figure(figsize=(16, 12))
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.3, wspace=0.3)

    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(t_full, f_full, "k.", ms=0.5, alpha=0.3, rasterized=True)
    if "t_zoom" in transit_result:
        zt = transit_result["t_zoom"]
        zf = transit_result["f_zoom"]
        ax1.plot(zt, zf, "r.", ms=1, alpha=0.5)
        ax1.axvline(zt.min(), color="orange", ls="--", alpha=0.5)
        ax1.axvline(zt.max(), color="orange", ls="--", alpha=0.5)
    ax1.set_xlabel("Time (BTJD)", fontsize=12); ax1.set_ylabel("Normalised Flux", fontsize=12)
    ax1.set_title(f"TIC {tic_id} — Full Light Curve", fontweight="bold")

    ax2 = fig.add_subplot(gs[1, 0])
    period = transit_result["period"]; epoch = transit_result["epoch"]
    phase = ((t_full - epoch + 0.5 * period) % period / period) - 0.5
    ax2.plot(phase, f_full, "k.", ms=0.8, alpha=0.2, rasterized=True)
    bins = np.linspace(-0.5, 0.5, 151)
    bc = 0.5 * (bins[:-1] + bins[1:])
    bidx = np.digitize(phase, bins[:-1])
    bf = np.array([np.nanmedian(f_full[bidx==i]) for i in range(1, len(bins))])
    ax2.plot(bc, bf, "r-", lw=2)
    ax2.set_xlabel("Phase"); ax2.set_ylabel("Flux")
    ax2.set_title("Phase-folded Curve"); ax2.set_xlim(-0.5, 0.5)

    ax3 = fig.add_subplot(gs[1, 1])
    if "t_zoom" in transit_result and "model_flux" in transit_result:
        ax3.plot(transit_result["t_zoom"], transit_result["f_zoom"], "k.", ms=2, alpha=0.5)
        ax3.plot(transit_result["t_zoom"], transit_result["model_flux"], "r-", lw=2)
    ax3.set_xlabel("Time (BTJD)"); ax3.set_ylabel("Flux")
    ax3.set_title("Transit + BATMAN Fit")

    ax4 = fig.add_subplot(gs[1, 2])
    if "t_zoom" in transit_result and "model_flux" in transit_result:
        res = transit_result["f_zoom"] - transit_result["model_flux"]
        ax4.plot(transit_result["t_zoom"], res, "k.", ms=2, alpha=0.5)
        ax4.axhline(0, color="r", ls="--")
    ax4.set_xlabel("Time (BTJD)"); ax4.set_ylabel("Residual")
    ax4.set_title(f"RMSE={transit_result.get('rmse', np.nan):.6f}")

    ax5 = fig.add_subplot(gs[2, 0])
    phys = estimate_physical_parameters(transit_result)
    text_items = [
        f"P = {phys.get('orbital_period_days', np.nan):.4f} d",
        f"R_p = {phys.get('planet_radius_rearth', np.nan):.2f} R_Earth",
        f"Depth = {phys.get('transit_depth_ppm', np.nan):.1f} ppm",
        f"b = {phys.get('impact_parameter', np.nan):.3f}",
        f"T_eq = {phys.get('equilibrium_temperature_k', np.nan):.0f} K",
        f"a/R_s = {phys.get('semi_major_axis_rs', np.nan):.2f}",
    ]
    ax5.axis("off")
    for i, item in enumerate(text_items):
        ax5.text(0.1, 0.9 - i * 0.15, item, transform=ax5.transAxes, fontsize=11,
                 fontfamily="monospace", verticalalignment="top")

    ax6 = fig.add_subplot(gs[2, 1:])
    if "t_zoom" in transit_result and "model_flux" in transit_result:
        res = transit_result["f_zoom"] - transit_result["model_flux"]
        ax6.hist(res, bins=25, color="steelblue", edgecolor="k", alpha=0.7)
        ax6.axvline(0, color="r", ls="--")
        ax6.set_xlabel("Residual Flux"); ax6.set_ylabel("Count")
        ax6.set_title(f"Residuals σ={np.std(res):.6f}, χ²_red={transit_result.get('red_chi2', np.nan):.3f}")

    plt.savefig(O_FIGURES / f"publication_panel_{tic_id}.png", dpi=300, bbox_inches="tight")
    plt.savefig(O_FIGURES / f"publication_panel_{tic_id}.pdf", bbox_inches="tight")
    plt.savefig(O_FIGURES / f"publication_panel_{tic_id}.svg", bbox_inches="tight")
    plt.show()

for _, row in transit_results.head(3).iterrows():
    lc = get_lightcurve(int(row["tic_id"]))
    create_publication_figure_panel(int(row["tic_id"]), lc, row)


# Part 10 — Export Results

Export all parameters, models, and reports in multiple formats.


In [114]:
def export_all_results(physical_df, transit_df, quality_df, mcmc_df=None):
    # Candidate parameters
    physical_df.to_parquet(O_CANDIDATE / "candidate_parameters.parquet", index=False)
    physical_df.to_csv(O_CANDIDATE / "candidate_parameters.csv", index=False)
    with open(O_CANDIDATE / "candidate_parameters.json", "w") as f:
        json.dump(physical_df.replace([np.inf, -np.inf], np.nan).fillna(np.nan).to_dict(orient="records"),
                  f, indent=2, default=str)
    logger.info("Exported candidate_parameters (.parquet, .csv, .json)")

    # Best-fit models
    model_rows = []
    for _, row in transit_df.iterrows():
        model_rows.append({
            "tic_id": int(row["tic_id"]),
            "period": row.get("period", np.nan),
            "epoch": row.get("epoch", np.nan),
            "rp_rs": row.get("rp_rs", np.nan),
            "inclination": row.get("inclination", np.nan),
            "a_rs": row.get("a_rs", np.nan),
            "red_chi2": row.get("red_chi2", np.nan),
            "rmse": row.get("rmse", np.nan),
            "r2": row.get("r2", np.nan),
            "bic": row.get("bic", np.nan),
            "aic": row.get("aic", np.nan),
        })
    model_df = pd.DataFrame(model_rows)
    model_df.to_parquet(O_MODELS / "best_fit_models.parquet", index=False)
    model_df.to_csv(O_MODELS / "best_fit_models.csv", index=False)
    logger.info("Exported best_fit_models")

    # Posterior statistics
    if mcmc_df is not None and len(mcmc_df) > 0:
        mcmc_df.to_parquet(O_MCMC / "posterior_statistics.parquet", index=False)
        mcmc_df.to_csv(O_MCMC / "posterior_statistics.csv", index=False)
        logger.info("Exported posterior_statistics")

    # Quality scores
    quality_df.to_parquet(O_CANDIDATE / "quality_scores.parquet", index=False)
    quality_df.to_csv(O_CANDIDATE / "quality_scores.csv", index=False)
    logger.info("Exported quality_scores")

    # Transit fit results
    transit_fit_cols = ["tic_id", "period", "epoch", "rp_rs", "inclination", "a_rs",
                        "red_chi2", "rmse", "r2", "bic"]
    available = [c for c in transit_fit_cols if c in transit_df.columns]
    if available:
        transit_df[available].to_csv(O_CANDIDATE / "transit_fit_results.csv", index=False)
        transit_df[available].to_parquet(O_CANDIDATE / "transit_fit_results.parquet", index=False)
        logger.info("Exported transit_fit_results")

    logger.info("All results exported successfully")
    return True

export_all_results(physical_df, transit_results, quality_df, mcmc_df if len(mcmc_df) > 0 else None)


2026-07-28 00:42:42,197 | INFO | Exported candidate_parameters (.parquet, .csv, .json)
2026-07-28 00:42:42,209 | INFO | Exported best_fit_models
2026-07-28 00:42:42,217 | INFO | Exported posterior_statistics
2026-07-28 00:42:42,227 | INFO | Exported quality_scores
2026-07-28 00:42:42,236 | INFO | Exported transit_fit_results
2026-07-28 00:42:42,236 | INFO | All results exported successfully


True

In [115]:
def generate_research_summary(physical_df, quality_df, transit_df):
    lines = []
    lines.append("# Transit Parameter Estimation — Research Summary")
    lines.append(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    lines.append("")
    lines.append("## Overview")
    lines.append(f"")
    lines.append(f"- Total candidates analysed: {len(physical_df)}")
    lines.append(f"- Candidates with successful fits: {(transit_df.get('success', pd.Series([True]*len(transit_df))).sum()) if 'success' in transit_df.columns else len(transit_df)}")
    lines.append(f"- MCMC uncertainty estimated: {len(mcmc_df) if len(mcmc_df) > 0 else 0}")
    lines.append("")

    lines.append("## Candidate Validation")
    for flag in ["CONFIRMED", "PROMISING", "UNCERTAIN"]:
        count = (quality_df["validation_flag"] == flag).sum() if "validation_flag" in quality_df.columns else 0
        lines.append(f"- **{flag}:** {count}")
    lines.append("")

    lines.append("## Physical Parameter Summary")
    for col in ["planet_radius_rearth", "orbital_period_days", "equilibrium_temperature_k", "transit_depth_ppm"]:
        if col in physical_df.columns:
            data = physical_df[col].replace([np.inf, -np.inf], np.nan).dropna()
            if len(data) > 0:
                lines.append(f"- **{col}:** median={data.median():.4f}, mean={data.mean():.4f}, "
                             f"min={data.min():.4f}, max={data.max():.4f}")
    lines.append("")

    lines.append("## Top Candidates")
    if "quality_score" in quality_df.columns:
        top = quality_df.sort_values("quality_score", ascending=False).head(3)
        for _, r in top.iterrows():
            lines.append(f"- TIC {int(r['tic_id'])}: score={r['quality_score']:.3f}, flag={r.get('validation_flag', 'N/A')}")
    lines.append("")

    lines.append("## Fit Quality")
    if "red_chi2" in transit_df.columns:
        rc = transit_df["red_chi2"].replace([np.inf, -np.inf], np.nan).dropna()
        lines.append(f"- Median reduced χ²: {rc.median():.3f}")
        lines.append(f"- Range: [{rc.min():.3f}, {rc.max():.3f}]")
    if "r2" in transit_df.columns:
        r2 = transit_df["r2"].replace([np.inf, -np.inf], np.nan).dropna()
        lines.append(f"- Median R²: {r2.median():.4f}")

    lines.append("")
    lines.append("---")
    lines.append("Generated by Notebook 08 — Transit Parameter Estimation")
    lines.append("AI-enabled Exoplanet Detection Pipeline")

    report = "\n".join(lines)

    with open(O_REPORTS / "research_summary.md", "w", encoding="utf-8") as f:
        f.write(report)
    logger.info("Research summary saved")

    # Also save as analysis report JSON
    summary = {
        "timestamp": datetime.now().isoformat(),
        "n_candidates": len(physical_df),
        "n_validated": int((quality_df["quality_score"] >= 0.7).sum()) if "quality_score" in quality_df.columns else 0,
        "n_promising": int(((quality_df["quality_score"] >= 0.4) & (quality_df["quality_score"] < 0.7)).sum()) if "quality_score" in quality_df.columns else 0,
        "top_candidates": [],
        "fit_statistics": {
            "median_red_chi2": float(transit_df["red_chi2"].median()) if "red_chi2" in transit_df.columns else None,
            "median_r2": float(transit_df["r2"].median()) if "r2" in transit_df.columns else None,
        }
    }
    if "quality_score" in quality_df.columns:
        for _, r in quality_df.sort_values("quality_score", ascending=False).head(5).iterrows():
            summary["top_candidates"].append({"tic_id": int(r["tic_id"]), "quality_score": float(r["quality_score"])})

    with open(O_REPORTS / "analysis_report.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    logger.info("Analysis report saved as JSON")
    return report

report_text = generate_research_summary(physical_df, quality_df, transit_results)
print(report_text[:1000])


2026-07-28 00:42:42,306 | INFO | Research summary saved
2026-07-28 00:42:42,313 | INFO | Analysis report saved as JSON


# Transit Parameter Estimation — Research Summary
**Generated:** 2026-07-28 00:42:42

## Overview

- Total candidates analysed: 23
- Candidates with successful fits: 23
- MCMC uncertainty estimated: 3

## Candidate Validation
- **CONFIRMED:** 1
- **PROMISING:** 22
- **UNCERTAIN:** 0

## Physical Parameter Summary
- **planet_radius_rearth:** median=0.6830, mean=0.6968, min=0.0953, max=2.0053
- **orbital_period_days:** median=3.4498, mean=4.7055, min=0.6074, max=13.2815
- **equilibrium_temperature_k:** median=1037.2746, mean=1136.4716, min=416.4997, max=2173.7092
- **transit_depth_ppm:** median=4792.7221, mean=6893.8368, min=92.1948, max=42380.8870

## Top Candidates
- TIC 152476657: score=0.715, flag=CONFIRMED
- TIC 288246496: score=0.697, flag=PROMISING
- TIC 460396820: score=0.685, flag=PROMISING

## Fit Quality
- Median reduced χ²: 1.003
- Range: [1.001, 1.086]
- Median R²: 0.0110

---
Generated by Notebook 08 — Transit Parameter Estimation
AI-enabled Exoplanet Detection Pipeline


In [116]:
def verify_export_files():
    expected = {
        "candidate_parameters.parquet": O_CANDIDATE,
        "candidate_parameters.csv": O_CANDIDATE,
        "candidate_parameters.json": O_CANDIDATE,
        "best_fit_models.parquet": O_MODELS,
        "best_fit_models.csv": O_MODELS,
        "quality_scores.parquet": O_CANDIDATE,
        "quality_scores.csv": O_CANDIDATE,
        "transit_fit_results.csv": O_CANDIDATE,
        "transit_fit_results.parquet": O_CANDIDATE,
        "mcmc_summary.parquet": O_MCMC,
        "research_summary.md": O_REPORTS,
        "analysis_report.json": O_REPORTS,
    }
    results = []
    for fname, dirpath in expected.items():
        fpath = dirpath / fname
        exists = fpath.exists()
        size = fpath.stat().st_size if exists else 0
        results.append({"file": fname, "exists": exists, "size_kb": size / 1024})
        status = "OK" if exists else "MISSING"
        logger.info(f"  [{status}] {fname} ({size/1024:.1f} KB)")
    return pd.DataFrame(results)

export_verification = verify_export_files()
export_verification


2026-07-28 00:42:42,348 | INFO |   [OK] candidate_parameters.parquet (17.9 KB)
2026-07-28 00:42:42,351 | INFO |   [OK] candidate_parameters.csv (8.6 KB)
2026-07-28 00:42:42,352 | INFO |   [OK] candidate_parameters.json (22.4 KB)
2026-07-28 00:42:42,353 | INFO |   [OK] best_fit_models.parquet (8.0 KB)
2026-07-28 00:42:42,354 | INFO |   [OK] best_fit_models.csv (4.5 KB)
2026-07-28 00:42:42,354 | INFO |   [OK] quality_scores.parquet (15.0 KB)
2026-07-28 00:42:42,355 | INFO |   [OK] quality_scores.csv (7.1 KB)
2026-07-28 00:42:42,356 | INFO |   [OK] transit_fit_results.csv (4.1 KB)
2026-07-28 00:42:42,356 | INFO |   [OK] transit_fit_results.parquet (7.4 KB)
2026-07-28 00:42:42,356 | INFO |   [OK] mcmc_summary.parquet (14.7 KB)
2026-07-28 00:42:42,358 | INFO |   [OK] research_summary.md (1.0 KB)
2026-07-28 00:42:42,360 | INFO |   [OK] analysis_report.json (0.7 KB)


,file,exists,size_kb
0,candidate_parameters.parquet,True,17.901367
1,candidate_parameters.csv,True,8.564453
2,candidate_parameters.json,True,22.437500
3,best_fit_models.parquet,True,8.008789
4,best_fit_models.csv,True,4.547852
5,quality_scores.parquet,True,14.985352
6,quality_scores.csv,True,7.104492
7,transit_fit_results.csv,True,4.136719
8,transit_fit_results.parquet,True,7.352539
9,mcmc_summary.parquet,True,14.692383


In [117]:
def compute_final_candidate_ranking(physical_df, quality_df, confidence_df):
    ranking = physical_df[["tic_id", "orbital_period_days", "planet_radius_rearth",
                           "equilibrium_temperature_k", "transit_depth_ppm"]].copy()
    ranking = ranking.merge(quality_df[["tic_id", "quality_score", "validation_flag"]], on="tic_id", how="left")
    ranking = ranking.merge(confidence_df[["tic_id", "final_confidence"]], on="tic_id", how="left")
    ranking["combined_score"] = ranking[["quality_score", "final_confidence"]].mean(axis=1)
    ranking = ranking.sort_values("combined_score", ascending=False).reset_index(drop=True)
    ranking["rank"] = range(1, len(ranking) + 1)
    ranking["priority"] = ranking["combined_score"].apply(
        lambda x: "HIGH" if x >= 0.7 else "MEDIUM" if x >= 0.4 else "LOW"
    )
    return ranking

final_ranking = compute_final_candidate_ranking(physical_df, quality_df, confidence_df)
final_ranking[["rank", "tic_id", "orbital_period_days", "planet_radius_rearth",
               "equilibrium_temperature_k", "combined_score", "priority"]]


,rank,tic_id,orbital_period_days,planet_radius_rearth,equilibrium_temperature_k,combined_score,priority
0,1,152476657,3.610647,1.024802,1539.640512,0.765221,HIGH
1,2,288246496,4.304908,0.788196,1307.908686,0.747395,HIGH
2,3,460396820,2.340504,1.360612,1037.274607,0.734752,HIGH
3,4,375654303,1.802074,1.200260,1841.931019,0.727892,HIGH
4,5,137637249,1.017064,0.808157,996.821513,0.722177,HIGH
...,...,...,...,...,...,...,...
78,79,100990000,9.600721,0.166897,674.330056,0.656347,MEDIUM
79,80,100990000,9.600721,0.166897,674.330056,0.656347,MEDIUM
80,81,100990000,9.600721,0.166897,674.330056,0.656347,MEDIUM
81,82,100990000,9.600721,0.166897,674.330056,0.656347,MEDIUM


In [118]:
def export_final_ranking(final_ranking):
    final_ranking.to_csv(O_CANDIDATE / "final_candidate_ranking.csv", index=False)
    final_ranking.to_parquet(O_CANDIDATE / "final_candidate_ranking.parquet", index=False)
    with open(O_CANDIDATE / "final_candidate_ranking.json", "w") as f:
        ranking_json = final_ranking.replace([np.inf, -np.inf], np.nan).fillna(np.nan).to_dict(orient="records")
        json.dump(ranking_json, f, indent=2, default=str)
    logger.info(f"Final ranking exported: {len(final_ranking)} candidates")
    logger.info(f"  HIGH priority: {(final_ranking['priority']=='HIGH').sum()}")
    logger.info(f"  MEDIUM priority: {(final_ranking['priority']=='MEDIUM').sum()}")
    logger.info(f"  LOW priority: {(final_ranking['priority']=='LOW').sum()}")

export_final_ranking(final_ranking)


2026-07-28 00:42:42,480 | INFO | Final ranking exported: 83 candidates
2026-07-28 00:42:42,480 | INFO |   HIGH priority: 52
2026-07-28 00:42:42,480 | INFO |   MEDIUM priority: 31
2026-07-28 00:42:42,487 | INFO |   LOW priority: 0


In [119]:
def export_all_figures_list():
    figure_files = sorted(O_FIGURES.glob("*"))
    logger.info(f"Total figures generated: {len(figure_files)}")
    for fmt in ["png", "pdf", "svg"]:
        n = len(list(O_FIGURES.glob(f"*.{fmt}")))
        logger.info(f"  .{fmt}: {n}")
    return figure_files

all_figures = export_all_figures_list()


2026-07-28 00:42:42,511 | INFO | Total figures generated: 156
2026-07-28 00:42:42,513 | INFO |   .png: 70
2026-07-28 00:42:42,513 | INFO |   .pdf: 47
2026-07-28 00:42:42,513 | INFO |   .svg: 36


In [120]:
def compute_summary_statistics(physical_df, quality_df, transit_df):
    stats = {
        "n_candidates": len(physical_df),
        "n_confirmed": int((quality_df["validation_flag"] == "CONFIRMED").sum()),
        "n_promising": int((quality_df["validation_flag"] == "PROMISING").sum()),
        "n_uncertain": int((quality_df["validation_flag"] == "UNCERTAIN").sum()),
        "median_period": float(physical_df["orbital_period_days"].median()),
        "median_radius": float(physical_df["planet_radius_rearth"].median()),
        "median_teq": float(physical_df["equilibrium_temperature_k"].median()),
        "median_chi2": float(transit_df["red_chi2"].median()),
        "median_quality": float(quality_df["quality_score"].median()),
        "n_mcmc": len(mcmc_df) if len(mcmc_df) > 0 else 0,
    }
    with open(O_REPORTS / "summary_statistics.json", "w") as f:
        json.dump(stats, f, indent=2, default=str)
    logger.info("Summary statistics exported")
    for k, v in stats.items(): logger.info(f"  {k}: {v}")
    return stats

final_stats = compute_summary_statistics(physical_df, quality_df, transit_results)


2026-07-28 00:42:42,551 | INFO | Summary statistics exported
2026-07-28 00:42:42,551 | INFO |   n_candidates: 23
2026-07-28 00:42:42,551 | INFO |   n_confirmed: 1
2026-07-28 00:42:42,555 | INFO |   n_promising: 22
2026-07-28 00:42:42,555 | INFO |   n_uncertain: 0
2026-07-28 00:42:42,557 | INFO |   median_period: 3.4497911845580957
2026-07-28 00:42:42,560 | INFO |   median_radius: 0.6830222169649157
2026-07-28 00:42:42,560 | INFO |   median_teq: 1037.274607118726
2026-07-28 00:42:42,562 | INFO |   median_chi2: 1.0033328525056815
2026-07-28 00:42:42,563 | INFO |   median_quality: 0.6700365359255127
2026-07-28 00:42:42,565 | INFO |   n_mcmc: 3


In [121]:
def plot_quality_vs_radius(physical_df, quality_df, save=True):
    merged = physical_df.merge(quality_df[["tic_id", "quality_score", "validation_flag"]], on="tic_id")
    fig, ax = plt.subplots(figsize=(9, 6))
    colors = {"CONFIRMED": "green", "PROMISING": "orange", "UNCERTAIN": "red"}
    for flag in ["CONFIRMED", "PROMISING", "UNCERTAIN"]:
        subset = merged[merged["validation_flag"] == flag]
        if len(subset) > 0:
            ax.scatter(subset["planet_radius_rearth"], subset["quality_score"],
                       c=colors[flag], s=60, label=flag, edgecolors="k", alpha=0.8)
    ax.set_xlabel("Planet Radius (R_Earth)"); ax.set_ylabel("Quality Score")
    ax.set_title("Quality Score vs Planet Radius", fontweight="bold"); ax.legend()
    if save:
        fig.savefig(O_FIGURES / "quality_vs_radius.png", dpi=300)
        fig.savefig(O_FIGURES / "quality_vs_radius.pdf")
    plt.show()

def plot_period_epoch_scatter(transit_results, save=True):
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.scatter(transit_results["period"], transit_results["epoch"], c=transit_results["red_chi2"],
               s=60, cmap="viridis_r", edgecolors="k", alpha=0.8)
    ax.set_xlabel("Orbital Period (days)"); ax.set_ylabel("Transit Epoch (BTJD)")
    ax.set_title("Period-Epoch Diagram", fontweight="bold"); ax.set_xscale("log")
    cb = plt.colorbar(ax.collections[0], ax=ax, label="Reduced chi2")
    if save:
        fig.savefig(O_FIGURES / "period_epoch_diagram.png", dpi=300)
        fig.savefig(O_FIGURES / "period_epoch_diagram.pdf")
    plt.show()

plot_quality_vs_radius(physical_df, quality_df)
plot_period_epoch_scatter(transit_results)


In [122]:
logger.info("=" * 70)
logger.info("NOTEBOOK 08 — COMPLETE")
logger.info(f"Candidates processed: {len(transit_results)}")
logger.info(f"Physical parameters estimated: {len(physical_df)}")
logger.info(f"Quality scores computed: {len(quality_df)}")
if len(mcmc_df) > 0:
    logger.info(f"MCMC uncertainties: {len(mcmc_df)}")
    try:
        n_total_samples = sum(len(pd.read_parquet(f)) for f in O_MCMC.glob("mcmc_samples_*.parquet"))
        logger.info(f"Total MCMC posterior samples: {n_total_samples}")
    except Exception:
        pass
logger.info(f"Figures saved to: {O_FIGURES}")
logger.info(f"Reports saved to: {O_REPORTS}")
logger.info(f"Parameters saved to: {O_CANDIDATE}")
logger.info(f"MCMC samples saved to: {O_MCMC}")
logger.info("=" * 70)

# Summary display
summary = {
    "Candidates Analysed": len(transit_results),
    "Physical Parameters": len(physical_df),
    "Quality Scores": len(quality_df),
    "MCMC Runs": len(mcmc_df) if len(mcmc_df) > 0 else 0,
    "CONFIRMED": int((quality_df["validation_flag"] == "CONFIRMED").sum()) if "validation_flag" in quality_df.columns else 0,
    "PROMISING": int((quality_df["validation_flag"] == "PROMISING").sum()) if "validation_flag" in quality_df.columns else 0,
    "UNCERTAIN": int((quality_df["validation_flag"] == "UNCERTAIN").sum()) if "validation_flag" in quality_df.columns else 0,
}
for k, v in summary.items():
    logger.info(f"  {k}: {v}")


2026-07-28 00:42:43,806 | INFO | ======================================================================
2026-07-28 00:42:43,806 | INFO | NOTEBOOK 08 — COMPLETE
2026-07-28 00:42:43,806 | INFO | Candidates processed: 23
2026-07-28 00:42:43,809 | INFO | Physical parameters estimated: 23
2026-07-28 00:42:43,809 | INFO | Quality scores computed: 23
2026-07-28 00:42:43,813 | INFO | MCMC uncertainties: 3
2026-07-28 00:42:43,832 | INFO | Total MCMC posterior samples: 153600
2026-07-28 00:42:43,832 | INFO | Figures saved to: d:\MachineLearning\Exoplanet-detection\outputs\figures
2026-07-28 00:42:43,832 | INFO | Reports saved to: d:\MachineLearning\Exoplanet-detection\outputs\reports
2026-07-28 00:42:43,832 | INFO | Parameters saved to: d:\MachineLearning\Exoplanet-detection\outputs\candidate_parameters
2026-07-28 00:42:43,832 | INFO | MCMC samples saved to: d:\MachineLearning\Exoplanet-detection\outputs\mcmc
2026-07-28 00:42:43,832 | INFO | ======================================================

In [123]:
def compute_transit_shape_metrics(row):
    rp_rs = row.get("rp_rs", 0)
    inc = row.get("inclination", 90)
    a_rs = row.get("a_rs", 0)
    if a_rs == 0: return {"shape": "Unknown", "b": np.nan}
    b = a_rs * np.cos(np.radians(inc))
    if abs(b) < 0.7: shape = "Central transit"
    elif abs(b) < 1 - rp_rs: shape = "Near-central"
    elif abs(b) < 1 + rp_rs: shape = "Grazing transit"
    else: shape = "Miss"
    return {"shape": shape, "b": float(b), "grazing": "Grazing" in shape}

shape_info = []
for _, row in transit_results.iterrows():
    info = compute_transit_shape_metrics(row)
    info["tic_id"] = int(row["tic_id"])
    info["rp_rs"] = row.get("rp_rs", np.nan)
    shape_info.append(info)
shape_df = pd.DataFrame(shape_info)
shape_counts = shape_df["shape"].value_counts()
logger.info("Transit shape classification:")
for s, c in shape_counts.items(): logger.info(f"  {s}: {c}")
shape_df


2026-07-28 00:42:43,895 | INFO | Transit shape classification:
2026-07-28 00:42:43,896 | INFO |   Central transit: 23


,shape,b,grazing,tic_id,rp_rs
0,Central transit,1.731748e-15,False,158002130,0.026050
1,Central transit,1.050161e-15,False,460396820,0.098466
2,Central transit,1.736972e-01,False,152476657,0.062522
3,Central transit,1.906970e-05,False,27318774,0.093024
4,Central transit,2.291641e-05,False,262662119,0.085994
5,Central transit,5.728389e-02,False,375654303,0.067401
6,Central transit,3.711149e-01,False,288246496,0.069229
7,Central transit,2.975353e-06,False,273373582,0.076546
8,Central transit,2.818171e-15,False,272080244,0.205866
9,Central transit,3.936633e-01,False,100990000,0.015355


In [124]:
if len(shape_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    shape_df["shape"].value_counts().plot(kind="bar", ax=axes[0], color=["steelblue","coral","gold","gray"], edgecolor="k")
    axes[0].set_xlabel("Shape"); axes[0].set_ylabel("Count"); axes[0].set_title("Transit Shape Classification")
    axes[0].tick_params(axis="x", rotation=30)
    axes[1].scatter(shape_df["rp_rs"], shape_df["b"], c=shape_df["grazing"].astype(int), s=60, cmap="RdYlGn_r", edgecolors="k", alpha=0.8)
    axes[1].axhline(0, color="gray", ls="--", alpha=0.5)
    axes[1].axhline(1, color="orange", ls=":", alpha=0.5, label="b=1")
    axes[1].set_xlabel("Rp/Rs"); axes[1].set_ylabel("Impact Parameter b")
    axes[1].set_title("Impact Parameter vs Radius Ratio"); axes[1].legend()
    plt.tight_layout()
    plt.savefig(O_FIGURES / "transit_shape_classification.png", dpi=150, bbox_inches="tight")
    plt.show()


In [125]:
def generate_scientific_report(physical_df, quality_df, transit_df):
    report_lines = []
    report_lines.append("=" * 60)
    report_lines.append("EXOPLANET TRANSIT PARAMETER ESTIMATION REPORT")
    report_lines.append("=" * 60)
    n_candidates = len(physical_df)
    n_confirmed = (quality_df["validation_flag"] == "CONFIRMED").sum() if "validation_flag" in quality_df.columns else 0
    n_promising = (quality_df["validation_flag"] == "PROMISING").sum() if "validation_flag" in quality_df.columns else 0
    report_lines.append(f"\nTotal candidates analyzed: {n_candidates}")
    report_lines.append(f"  Confirmed: {n_confirmed}")
    report_lines.append(f"  Promising: {n_promising}")
    report_lines.append(f"  Uncertain: {n_candidates - n_confirmed - n_promising}")
    avg_quality = quality_df["quality_score"].mean()
    report_lines.append(f"\nAverage quality score: {avg_quality:.3f}")
    avg_rp = physical_df["planet_radius_rearth"].mean()
    report_lines.append(f"Average planet radius: {avg_rp:.2f} R_Earth")
    avg_teq = physical_df["equilibrium_temperature_k"].mean()
    report_lines.append(f"Average equilibrium temperature: {avg_teq:.0f} K")
    n_mcmc = len(transit_df[transit_df["mcmc_complete"] == True]) if "mcmc_complete" in transit_df.columns else 0
    report_lines.append(f"\nMCMC complete: {n_mcmc}")
    report_lines.append("\nTop 5 candidates by quality:")
    top5 = quality_df.sort_values("quality_score", ascending=False).head(5)
    for _, r in top5.iterrows():
        report_lines.append(f"  TIC {int(r['tic_id'])}: score={r['quality_score']:.3f} ({r.get('validation_flag', 'N/A')})")
    report_lines.append("\n" + "=" * 60)
    report_str = "\n".join(report_lines)
    (O_REPORTS / "science_report.txt").write_text(report_str)
    logger.info(report_str)
    return report_str

_sci_report = generate_scientific_report(physical_df, quality_df, transit_results) if len(physical_df) > 0 and len(quality_df) > 0 else ""


2026-07-28 00:42:44,471 | INFO | ============================================================
EXOPLANET TRANSIT PARAMETER ESTIMATION REPORT

Total candidates analyzed: 23
  Confirmed: 1
  Promising: 22
  Uncertain: 0

Average quality score: 0.658
Average planet radius: 0.70 R_Earth
Average equilibrium temperature: 1136 K

MCMC complete: 0

Top 5 candidates by quality:
  TIC 152476657: score=0.715 (CONFIRMED)
  TIC 288246496: score=0.697 (PROMISING)
  TIC 460396820: score=0.685 (PROMISING)
  TIC 375654303: score=0.678 (PROMISING)
  TIC 137637249: score=0.672 (PROMISING)



In [126]:
def compute_final_data_product_summary():
    summary = {
        "candidate_params": list(O_CANDIDATE.glob("*")),
        "transit_models": list(O_MODELS.glob("*")),
        "figures": list(O_FIGURES.glob("*")),
        "mcmc_samples": list(O_MCMC.glob("*")),
        "reports": list(O_REPORTS.glob("*")),
    }
    logger.info("Data product summary:")
    for category, files in summary.items():
        logger.info(f"  {category}: {len(files)} files")
    total = sum(len(v) for v in summary.values())
    logger.info(f"  TOTAL: {total} files generated")
    return summary

data_summary = compute_final_data_product_summary()


2026-07-28 00:42:44,496 | INFO | Data product summary:
2026-07-28 00:42:44,498 | INFO |   candidate_params: 11 files
2026-07-28 00:42:44,498 | INFO |   transit_models: 4 files
2026-07-28 00:42:44,498 | INFO |   figures: 156 files
2026-07-28 00:42:44,498 | INFO |   mcmc_samples: 9 files
2026-07-28 00:42:44,498 | INFO |   reports: 4 files
2026-07-28 00:42:44,503 | INFO |   TOTAL: 184 files generated


In [127]:
# Export list of all generated files
def list_generated_files():
    files = []
    for root, dirs, fnames in os.walk(OUTPUTS_DIR):
        for fname in fnames:
            fpath = Path(root) / fname
            files.append(str(fpath.relative_to(PROJECT_ROOT)))
    return sorted(files)

generated = list_generated_files()
logger.info(f"\nGenerated {len(generated)} files:")
for f in generated:
    logger.info(f"  └─ {f}")


2026-07-28 00:42:44,563 | INFO | 
Generated 270 files:
2026-07-28 00:42:44,563 | INFO |   └─ outputs\candidate_parameters\candidate_parameters.csv
2026-07-28 00:42:44,563 | INFO |   └─ outputs\candidate_parameters\candidate_parameters.json
2026-07-28 00:42:44,563 | INFO |   └─ outputs\candidate_parameters\candidate_parameters.parquet
2026-07-28 00:42:44,563 | INFO |   └─ outputs\candidate_parameters\candidate_summary.csv
2026-07-28 00:42:44,563 | INFO |   └─ outputs\candidate_parameters\final_candidate_ranking.csv
2026-07-28 00:42:44,563 | INFO |   └─ outputs\candidate_parameters\final_candidate_ranking.json
2026-07-28 00:42:44,563 | INFO |   └─ outputs\candidate_parameters\final_candidate_ranking.parquet
2026-07-28 00:42:44,563 | INFO |   └─ outputs\candidate_parameters\quality_scores.csv
2026-07-28 00:42:44,563 | INFO |   └─ outputs\candidate_parameters\quality_scores.parquet
2026-07-28 00:42:44,563 | INFO |   └─ outputs\candidate_parameters\transit_fit_results.csv
2026-07-28 00:42:4